In [1]:
# Cell 1 — Verify QML environment
# Confirms that this notebook is using the separate Sepsis QML Python environment and expected Qiskit versions.

import sys
import qiskit
import qiskit_machine_learning

print("Python executable:", sys.executable)
print("Qiskit version:", qiskit.__version__)
print("Qiskit ML version:", qiskit_machine_learning.__version__)

Python executable: d:\SepsiChronos HDA\.venv\Scripts\python.exe
Qiskit version: 2.5.2
Qiskit ML version: 0.9.1


In [2]:
# Cell 2 — Create a 1-qubit quantum circuit
# Applies a Hadamard gate to place the qubit into an equal superposition of |0> and |1>.

from qiskit import QuantumCircuit

qc = QuantumCircuit(1)
qc.h(0)

print(qc.draw("text"))

   ┌───┐
q: ┤ H ├
   └───┘


In [3]:
# Cell 3 — Simulate and inspect the quantum state
# Calculates the statevector and verifies the expected 50%-50% measurement probabilities.

from qiskit.quantum_info import Statevector

state = Statevector.from_instruction(qc)

print("Statevector:")
print(state)

print("\nMeasurement probabilities:")
print(state.probabilities_dict())

Statevector:
Statevector([0.70710678+0.j, 0.70710678+0.j],
            dims=(2,))

Measurement probabilities:
{np.str_('0'): np.float64(0.4999999999999999), np.str_('1'): np.float64(0.4999999999999999)}


In [4]:
# Cell 4 — Create a 2-qubit entangled quantum circuit
# Uses a Hadamard gate followed by CNOT to create a Bell-state entangled pair.

from qiskit import QuantumCircuit

bell_qc = QuantumCircuit(2)

bell_qc.h(0)
bell_qc.cx(0, 1)

print(bell_qc.draw("text"))

     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘


In [5]:
# Cell 5 — Simulate and inspect the entangled Bell state
# Verifies that only |00> and |11> occur, demonstrating correlation between the two qubits.

bell_state = Statevector.from_instruction(bell_qc)

print("Bell-state statevector:")
print(bell_state)

print("\nMeasurement probabilities:")
print(bell_state.probabilities_dict())

Bell-state statevector:
Statevector([0.70710678+0.j, 0.        +0.j, 0.        +0.j,
             0.70710678+0.j],
            dims=(2, 2))

Measurement probabilities:
{np.str_('00'): np.float64(0.4999999999999999), np.str_('11'): np.float64(0.4999999999999999)}


In [6]:
# Cell 6 — Load saved Review 1 reference artifacts
# Loads the modelling objects exported from the completed Review 1 notebook for validation and QML reuse.

from pathlib import Path
import joblib

def find_project_root():
    here = Path.cwd()
    for folder in [here, here.parent]:
        if (folder / "notebooks").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from the repository root or from the notebooks/ folder.")

PROJECT_DIR = find_project_root()
HANDOFF_DIR = PROJECT_DIR / "data" / "processed"

model_data = joblib.load(HANDOFF_DIR / "model_data.joblib")
review1_scaler = joblib.load(HANDOFF_DIR / "preprocessing_scaler.joblib")

print("Review 1 model data loaded successfully.")
print("Review 1 preprocessing scaler loaded successfully.")

print("\nAvailable model_data keys:")
for key in model_data.keys():
    print("-", key)

Review 1 model data loaded successfully.
Review 1 preprocessing scaler loaded successfully.

Available model_data keys:
- X_train
- X_test
- y_train
- y_test
- model_features
- train_patient_keys
- test_patient_keys


In [7]:
# Cell 7 — Verify Review 1 reference data structure and patient split
# Confirms train/test shapes, feature count, target alignment, and zero patient overlap.

X_train_ref = model_data["X_train"]
X_test_ref = model_data["X_test"]
y_train_ref = model_data["y_train"]
y_test_ref = model_data["y_test"]
model_features_ref = model_data["model_features"]
train_patient_keys_ref = model_data["train_patient_keys"]
test_patient_keys_ref = model_data["test_patient_keys"]

print("=== DATA STRUCTURE ===")
print("X_train shape:", X_train_ref.shape)
print("X_test shape :", X_test_ref.shape)
print("y_train length:", len(y_train_ref))
print("y_test length :", len(y_test_ref))
print("Number of model features:", len(model_features_ref))

print("\n=== FEATURE CONSISTENCY ===")
print(
    "X_train columns match model_features:",
    list(X_train_ref.columns) == list(model_features_ref)
)
print(
    "X_test columns match model_features:",
    list(X_test_ref.columns) == list(model_features_ref)
)

train_patient_set = set(train_patient_keys_ref)
test_patient_set = set(test_patient_keys_ref)

print("\n=== PATIENT SPLIT ===")
print("Training patients:", len(train_patient_set))
print("Testing patients :", len(test_patient_set))
print("Patient overlap  :", len(train_patient_set.intersection(test_patient_set)))

print("\n=== ROW/TARGET ALIGNMENT ===")
print("Train rows match y_train:", len(X_train_ref) == len(y_train_ref))
print("Test rows match y_test  :", len(X_test_ref) == len(y_test_ref))

=== DATA STRUCTURE ===
X_train shape: (931285, 40)
X_test shape : (232513, 40)
y_train length: 931285
y_test length : 232513
Number of model features: 40

=== FEATURE CONSISTENCY ===
X_train columns match model_features: True
X_test columns match model_features: True

=== PATIENT SPLIT ===
Training patients: 24270
Testing patients : 6068
Patient overlap  : 0

=== ROW/TARGET ALIGNMENT ===
Train rows match y_train: True
Test rows match y_test  : True


In [8]:
# Cell 8 — Verify class distribution and missing values
# Confirms the frozen Review 1 reference data retains the expected class imbalance and contains no missing modelling values.

import pandas as pd

print("=== TRAIN TARGET DISTRIBUTION ===")
train_counts = pd.Series(y_train_ref).value_counts().sort_index()
train_pct = pd.Series(y_train_ref).value_counts(normalize=True).sort_index() * 100

for label in train_counts.index:
    print(
        f"Class {label}: {train_counts[label]:,} "
        f"({train_pct[label]:.4f}%)"
    )

print("\n=== TEST TARGET DISTRIBUTION ===")
test_counts = pd.Series(y_test_ref).value_counts().sort_index()
test_pct = pd.Series(y_test_ref).value_counts(normalize=True).sort_index() * 100

for label in test_counts.index:
    print(
        f"Class {label}: {test_counts[label]:,} "
        f"({test_pct[label]:.4f}%)"
    )

print("\n=== MISSING VALUES ===")
print("Missing values in X_train:", int(X_train_ref.isna().sum().sum()))
print("Missing values in X_test :", int(X_test_ref.isna().sum().sum()))
print("Missing values in y_train:", int(pd.Series(y_train_ref).isna().sum()))
print("Missing values in y_test :", int(pd.Series(y_test_ref).isna().sum()))

=== TRAIN TARGET DISTRIBUTION ===
Class 0: 915,552 (98.3106%)
Class 1: 15,733 (1.6894%)

=== TEST TARGET DISTRIBUTION ===
Class 0: 228,611 (98.3218%)
Class 1: 3,902 (1.6782%)

=== MISSING VALUES ===
Missing values in X_train: 0
Missing values in X_test : 0
Missing values in y_train: 0
Missing values in y_test : 0


In [9]:
# Cell 9 — Verify preprocessing scaler and feature metadata
# Confirms that the saved Review 1 scaler matches the expected continuous-feature preprocessing setup.

print("=== SCALER CHECK ===")
print("Scaler type:", type(review1_scaler).__name__)

if hasattr(review1_scaler, "n_features_in_"):
    print("Scaler input feature count:", review1_scaler.n_features_in_)

if hasattr(review1_scaler, "feature_names_in_"):
    print("Scaler feature-name count:", len(review1_scaler.feature_names_in_))
    print("First 10 scaler features:", list(review1_scaler.feature_names_in_[:10]))

print("\n=== MODEL FEATURE CHECK ===")
print("Total model features:", len(model_features_ref))
print("First 10 model features:", list(model_features_ref[:10]))

binary_features = ["Gender", "Unit1", "Unit2"]
continuous_features_ref = [
    f for f in model_features_ref
    if f not in binary_features
]

print("\nExpected continuous features:", len(continuous_features_ref))
print("Expected binary features:", len(binary_features))

if hasattr(review1_scaler, "n_features_in_"):
    print(
        "Scaler count matches expected continuous features:",
        review1_scaler.n_features_in_ == len(continuous_features_ref)
    )

if hasattr(review1_scaler, "feature_names_in_"):
    print(
        "Scaler feature names match expected continuous features:",
        list(review1_scaler.feature_names_in_) == continuous_features_ref
    )

=== SCALER CHECK ===
Scaler type: StandardScaler
Scaler input feature count: 37
Scaler feature-name count: 37
First 10 scaler features: ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3']

=== MODEL FEATURE CHECK ===
Total model features: 40
First 10 model features: ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3']

Expected continuous features: 37
Expected binary features: 3
Scaler count matches expected continuous features: True
Scaler feature names match expected continuous features: True


In [10]:
# Cell 10 — Inspect patient-key structure for QML dataset reduction
# Determines whether the saved patient keys provide row-level mapping or only unique patient identifiers.

print("=== TRAIN PATIENT KEY STRUCTURE ===")
print("Object type:", type(train_patient_keys_ref))
print("Number of stored train patient-key entries:", len(train_patient_keys_ref))
print("Number of X_train rows:", len(X_train_ref))
print("Unique train patient keys:", len(set(train_patient_keys_ref)))
print("Sample train patient keys:", list(train_patient_keys_ref)[:10])

print("\n=== TEST PATIENT KEY STRUCTURE ===")
print("Object type:", type(test_patient_keys_ref))
print("Number of stored test patient-key entries:", len(test_patient_keys_ref))
print("Number of X_test rows:", len(X_test_ref))
print("Unique test patient keys:", len(set(test_patient_keys_ref)))
print("Sample test patient keys:", list(test_patient_keys_ref)[:10])

print("\n=== ROW-LEVEL MAPPING CHECK ===")
print(
    "Train patient keys map one-to-one with X_train rows:",
    len(train_patient_keys_ref) == len(X_train_ref)
)
print(
    "Test patient keys map one-to-one with X_test rows:",
    len(test_patient_keys_ref) == len(X_test_ref)
)

=== TRAIN PATIENT KEY STRUCTURE ===
Object type: <class 'set'>
Number of stored train patient-key entries: 24270
Number of X_train rows: 931285
Unique train patient keys: 24270
Sample train patient keys: ['B_p104666', 'B_p100276', 'B_p117652', 'A_p018506', 'B_p107152', 'A_p017812', 'A_p017712', 'B_p103172', 'B_p107296', 'B_p109210']

=== TEST PATIENT KEY STRUCTURE ===
Object type: <class 'set'>
Number of stored test patient-key entries: 6068
Number of X_test rows: 232513
Unique test patient keys: 6068
Sample test patient keys: ['B_p116955', 'B_p107544', 'B_p102991', 'B_p112744', 'B_p118532', 'B_p114545', 'A_p011040', 'A_p017184', 'A_p012726', 'A_p011916']

=== ROW-LEVEL MAPPING CHECK ===
Train patient keys map one-to-one with X_train rows: False
Test patient keys map one-to-one with X_test rows: False


In [11]:
# Cell 11 — Inspect retained row indices in the Review 1 modelling data
# Checks whether train/test features and targets preserve original row indices that can be linked back to patient identities.

print("=== X_TRAIN INDEX ===")
print("Index type:", type(X_train_ref.index))
print("Number of rows:", len(X_train_ref.index))
print("Unique index values:", X_train_ref.index.nunique())
print("First 10 indices:", X_train_ref.index[:10].tolist())
print("Last 10 indices :", X_train_ref.index[-10:].tolist())

print("\n=== X_TEST INDEX ===")
print("Index type:", type(X_test_ref.index))
print("Number of rows:", len(X_test_ref.index))
print("Unique index values:", X_test_ref.index.nunique())
print("First 10 indices:", X_test_ref.index[:10].tolist())
print("Last 10 indices :", X_test_ref.index[-10:].tolist())

print("\n=== FEATURE/TARGET INDEX ALIGNMENT ===")
print("X_train and y_train indices identical:",
      X_train_ref.index.equals(y_train_ref.index))

print("X_test and y_test indices identical:",
      X_test_ref.index.equals(y_test_ref.index))

print("\n=== TRAIN/TEST INDEX OVERLAP ===")
print(
    "Number of shared dataframe indices:",
    len(set(X_train_ref.index).intersection(set(X_test_ref.index)))
)

=== X_TRAIN INDEX ===
Index type: <class 'pandas.RangeIndex'>
Number of rows: 931285
Unique index values: 931285
First 10 indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Last 10 indices : [931275, 931276, 931277, 931278, 931279, 931280, 931281, 931282, 931283, 931284]

=== X_TEST INDEX ===
Index type: <class 'pandas.RangeIndex'>
Number of rows: 232513
Unique index values: 232513
First 10 indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Last 10 indices : [232503, 232504, 232505, 232506, 232507, 232508, 232509, 232510, 232511, 232512]

=== FEATURE/TARGET INDEX ALIGNMENT ===
X_train and y_train indices identical: True
X_test and y_test indices identical: True

=== TRAIN/TEST INDEX OVERLAP ===
Number of shared dataframe indices: 232513


In [12]:
# Cell 12 — Locate Review 1 patient-split and modelling-data code
# Searches the completed Review 1 notebook for only the cells relevant to train/test patient mapping and feature construction.

import json

review1_notebook_path = PROJECT_DIR / "notebooks" / "01_EDA_and_Preprocessing.ipynb"

with open(review1_notebook_path, "r", encoding="utf-8") as f:
    review1_nb = json.load(f)

search_terms = [
    "train_patient_keys",
    "test_patient_keys",
    "train_df",
    "test_df",
    "X_train",
    "X_test",
    "Patient_Key"
]

for i, cell in enumerate(review1_nb["cells"]):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))

    if any(term in source for term in search_terms):
        print(f"\n{'='*80}")
        print(f"REVIEW 1 CODE CELL INDEX: {i}")
        print("="*80)
        print(source[:5000])


REVIEW 1 CODE CELL INDEX: 8
# CELL 9 — Create unique patient key and visualize class distribution

import matplotlib.pyplot as plt

# Guaranteed unique patient identifier
combined_df["Patient_Key"] = (
    combined_df["Hospital_Set"].astype(str)
    + "_"
    + combined_df["Patient_ID"].astype(str)
)

print("Unique Patient_Key count:")
print(combined_df["Patient_Key"].nunique())


# ----- Hour-level class distribution -----

hour_class_counts = combined_df["SepsisLabel"].value_counts().sort_index()

hour_class_percent = (
    combined_df["SepsisLabel"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print("\nHour-level class counts:")
print(hour_class_counts)

print("\nHour-level class percentages:")
print(hour_class_percent.round(2))


plt.figure(figsize=(6, 4))

plt.bar(
    ["No Sepsis Warning (0)", "Sepsis Warning (1)"],
    hour_class_counts.values
)

plt.title("Hourly SepsisLabel Distribution")
plt.ylabel("Number of Hourly Observations")
plt.xlabel("Class")

plt

In [13]:
# Cell 13 — Reconstruct row-level patient mapping from the raw reference data
# Recreates the exact Review 1 patient/hour ordering so every modelling row can be linked to its patient safely.

import numpy as np
import pandas as pd

RAW_DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "sepsis_combined_raw.csv"
)

mapping_cols = [
    "Hospital_Set",
    "Patient_ID",
    "ICULOS",
    "SepsisLabel"
]

raw_mapping = pd.read_csv(
    RAW_DATA_PATH,
    usecols=mapping_cols
)

# Recreate the exact unique patient identifier used in Review 1
raw_mapping["Patient_Key"] = (
    raw_mapping["Hospital_Set"].astype(str)
    + "_"
    + raw_mapping["Patient_ID"].astype(str)
)

# Reapply the already-frozen Review 1 patient split
train_row_map = raw_mapping[
    raw_mapping["Patient_Key"].isin(train_patient_keys_ref)
].copy()

test_row_map = raw_mapping[
    raw_mapping["Patient_Key"].isin(test_patient_keys_ref)
].copy()

# Reproduce the exact ordering used in Review 1 Cell 20
train_row_map = (
    train_row_map
    .sort_values(["Patient_Key", "ICULOS"])
    .reset_index(drop=True)
)

test_row_map = (
    test_row_map
    .sort_values(["Patient_Key", "ICULOS"])
    .reset_index(drop=True)
)

print("=== RECONSTRUCTED ROW MAPPING ===")
print("Train mapping rows:", len(train_row_map))
print("Test mapping rows :", len(test_row_map))

print("\nTrain unique patients:", train_row_map["Patient_Key"].nunique())
print("Test unique patients :", test_row_map["Patient_Key"].nunique())

print("\n=== ALIGNMENT WITH SAVED REVIEW 1 DATA ===")

print(
    "Train row count matches X_train:",
    len(train_row_map) == len(X_train_ref)
)

print(
    "Test row count matches X_test:",
    len(test_row_map) == len(X_test_ref)
)

print(
    "Train SepsisLabel sequence matches y_train:",
    np.array_equal(
        train_row_map["SepsisLabel"].to_numpy(),
        np.asarray(y_train_ref)
    )
)

print(
    "Test SepsisLabel sequence matches y_test:",
    np.array_equal(
        test_row_map["SepsisLabel"].to_numpy(),
        np.asarray(y_test_ref)
    )
)

print(
    "Train ICULOS sequence matches X_train:",
    np.array_equal(
        train_row_map["ICULOS"].to_numpy(),
        X_train_ref["ICULOS"].to_numpy()
    )
)

print(
    "Test ICULOS sequence matches X_test:",
    np.array_equal(
        test_row_map["ICULOS"].to_numpy(),
        X_test_ref["ICULOS"].to_numpy()
    )
)

# Freeze row-level patient identity aligned to X/y
train_row_patient_keys_ref = train_row_map["Patient_Key"].copy()
test_row_patient_keys_ref = test_row_map["Patient_Key"].copy()

print("\n=== PATIENT MAPPING VALIDATION ===")

print(
    "Recovered train patient set matches saved set:",
    set(train_row_patient_keys_ref) == train_patient_keys_ref
)

print(
    "Recovered test patient set matches saved set:",
    set(test_row_patient_keys_ref) == test_patient_keys_ref
)

print(
    "Recovered train/test patient overlap:",
    len(
        set(train_row_patient_keys_ref)
        .intersection(set(test_row_patient_keys_ref))
    )
)

=== RECONSTRUCTED ROW MAPPING ===
Train mapping rows: 931285
Test mapping rows : 232513

Train unique patients: 24270
Test unique patients : 6068

=== ALIGNMENT WITH SAVED REVIEW 1 DATA ===
Train row count matches X_train: True
Test row count matches X_test: True
Train SepsisLabel sequence matches y_train: True
Test SepsisLabel sequence matches y_test: True
Train ICULOS sequence matches X_train: True
Test ICULOS sequence matches X_test: True

=== PATIENT MAPPING VALIDATION ===
Recovered train patient set matches saved set: True
Recovered test patient set matches saved set: True
Recovered train/test patient overlap: 0


In [14]:
# Cell 14 — Profile sepsis-positive hours across patients
# Examines how positive and negative hourly observations are distributed across septic and non-septic patients before QML sampling.

def patient_class_profile(row_map, split_name):
    patient_summary = (
        row_map
        .groupby("Patient_Key")
        .agg(
            total_hours=("SepsisLabel", "size"),
            positive_hours=("SepsisLabel", "sum")
        )
    )

    patient_summary["negative_hours"] = (
        patient_summary["total_hours"]
        - patient_summary["positive_hours"]
    )

    patient_summary["patient_class"] = (
        patient_summary["positive_hours"] > 0
    ).astype(int)

    septic = patient_summary[
        patient_summary["patient_class"] == 1
    ]

    non_septic = patient_summary[
        patient_summary["patient_class"] == 0
    ]

    print(f"=== {split_name} PATIENT PROFILE ===")

    print("Total patients:", len(patient_summary))
    print("Septic patients:", len(septic))
    print("Non-septic patients:", len(non_septic))

    print("\nTotal positive hours:",
          int(patient_summary["positive_hours"].sum()))

    print("Total negative hours:",
          int(patient_summary["negative_hours"].sum()))

    print("\nPositive hours per septic patient:")
    print(
        septic["positive_hours"]
        .describe(
            percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        )
        .round(2)
    )

    print("\nTotal ICU hours per septic patient:")
    print(
        septic["total_hours"]
        .describe(
            percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        )
        .round(2)
    )

    return patient_summary


train_patient_profile = patient_class_profile(
    train_row_map,
    "TRAIN"
)

print("\n")

test_patient_profile = patient_class_profile(
    test_row_map,
    "TEST"
)

=== TRAIN PATIENT PROFILE ===
Total patients: 24270
Septic patients: 1654
Non-septic patients: 22616

Total positive hours: 15733
Total negative hours: 915552

Positive hours per septic patient:
count    1654.00
mean        9.51
std         0.84
min         1.00
25%         9.00
50%        10.00
75%        10.00
90%        10.00
95%        10.00
99%        10.00
max        10.00
Name: positive_hours, dtype: float64

Total ICU hours per septic patient:
count    1654.00
mean       59.20
std        59.78
min         8.00
25%        15.00
50%        38.00
75%        82.75
90%       144.00
95%       190.70
99%       263.94
max       336.00
Name: total_hours, dtype: float64


=== TEST PATIENT PROFILE ===
Total patients: 6068
Septic patients: 413
Non-septic patients: 5655

Total positive hours: 3902
Total negative hours: 228611

Positive hours per septic patient:
count    413.00
mean       9.45
std        1.01
min        1.00
25%        9.00
50%       10.00
75%       10.00
90%       10.00
95%

In [15]:
# Cell 15 — Profile sources of negative hourly observations
# Separates negative hours from septic and non-septic patients to guide patient-aware QML sampling.

def negative_source_profile(patient_summary, split_name):

    septic = patient_summary[
        patient_summary["patient_class"] == 1
    ]

    non_septic = patient_summary[
        patient_summary["patient_class"] == 0
    ]

    septic_negative_hours = int(
        septic["negative_hours"].sum()
    )

    non_septic_negative_hours = int(
        non_septic["negative_hours"].sum()
    )

    total_negative_hours = (
        septic_negative_hours
        + non_septic_negative_hours
    )

    print(f"=== {split_name} NEGATIVE-HOUR SOURCES ===")

    print(
        "Negative hours from septic patients:",
        f"{septic_negative_hours:,}"
    )

    print(
        "Negative hours from non-septic patients:",
        f"{non_septic_negative_hours:,}"
    )

    print(
        "Total negative hours:",
        f"{total_negative_hours:,}"
    )

    print("\nPercentage of negative pool:")

    print(
        "From septic patients:",
        f"{septic_negative_hours / total_negative_hours * 100:.2f}%"
    )

    print(
        "From non-septic patients:",
        f"{non_septic_negative_hours / total_negative_hours * 100:.2f}%"
    )

    print("\nSeptic patients with at least one negative hour:")
    print(int((septic["negative_hours"] > 0).sum()))

    print("Septic patients with zero negative hours:")
    print(int((septic["negative_hours"] == 0).sum()))

    print("\nNegative hours per SEPTIC patient:")
    print(
        septic["negative_hours"]
        .describe(
            percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        )
        .round(2)
    )

    print("\nNegative hours per NON-SEPTIC patient:")
    print(
        non_septic["negative_hours"]
        .describe(
            percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        )
        .round(2)
    )


negative_source_profile(
    train_patient_profile,
    "TRAIN"
)

print("\n")

negative_source_profile(
    test_patient_profile,
    "TEST"
)

=== TRAIN NEGATIVE-HOUR SOURCES ===
Negative hours from septic patients: 82,186
Negative hours from non-septic patients: 833,366
Total negative hours: 915,552

Percentage of negative pool:
From septic patients: 8.98%
From non-septic patients: 91.02%

Septic patients with at least one negative hour:
1393
Septic patients with zero negative hours:
261

Negative hours per SEPTIC patient:
count    1654.00
mean       49.69
std        59.68
min         0.00
25%         5.00
50%        29.00
75%        73.00
90%       135.00
95%       180.70
99%       254.47
max       330.00
Name: negative_hours, dtype: float64

Negative hours per NON-SEPTIC patient:
count    22616.00
mean        36.85
std         16.30
min          8.00
25%         24.00
50%         38.00
75%         46.00
90%         53.00
95%         56.00
99%         59.00
max        336.00
Name: negative_hours, dtype: float64


=== TEST NEGATIVE-HOUR SOURCES ===
Negative hours from septic patients: 19,919
Negative hours from non-septic pa

In [16]:
# Cell 16 — Compare candidate QML sample sizes
# Estimates patient diversity and positive-example coverage for several feasible reduced dataset sizes.

import pandas as pd
import numpy as np

train_candidate_sizes = [500, 1000, 1500, 2000]
test_candidate_sizes = [3000, 4000, 5000]

train_septic_patients = len(
    train_patient_profile[
        train_patient_profile["patient_class"] == 1
    ]
)

test_positive_prevalence = (
    int((y_test_ref == 1).sum()) / len(y_test_ref)
)

train_rows = []

for total_size in train_candidate_sizes:

    # Candidate training design: approximately balanced
    positive_rows = total_size // 2
    negative_rows = total_size - positive_rows

    septic_negative_rows = round(
        negative_rows * 0.0939
    )

    nonseptic_negative_rows = (
        negative_rows - septic_negative_rows
    )

    train_rows.append({
        "Train size": total_size,
        "Positive rows": positive_rows,
        "Negative rows": negative_rows,
        "Positive septic patients needed": positive_rows,
        "Septic-positive patient coverage (%)":
            positive_rows / train_septic_patients * 100,
        "Negative rows from septic patients":
            septic_negative_rows,
        "Negative rows from non-septic patients":
            nonseptic_negative_rows
    })

train_candidate_table = pd.DataFrame(train_rows)

test_rows = []

for total_size in test_candidate_sizes:

    positive_rows = round(
        total_size * test_positive_prevalence
    )

    negative_rows = total_size - positive_rows

    test_rows.append({
        "Test size": total_size,
        "Expected positive rows": positive_rows,
        "Expected negative rows": negative_rows,
        "Expected positive prevalence (%)":
            positive_rows / total_size * 100
    })

test_candidate_table = pd.DataFrame(test_rows)

print("=== TRAINING CANDIDATES ===")
display(train_candidate_table.round(2))

print("\n=== TEST CANDIDATES ===")
display(test_candidate_table.round(2))

=== TRAINING CANDIDATES ===


,Train size,Positive rows,Negative rows,Positive septic patients needed,Septic-positive patient coverage (%),Negative rows from septic patients,Negative rows from non-septic patients
0,500,250,250,250,15.11,23,227
1,1000,500,500,500,30.23,47,453
2,1500,750,750,750,45.34,70,680
3,2000,1000,1000,1000,60.46,94,906



=== TEST CANDIDATES ===


,Test size,Expected positive rows,Expected negative rows,Expected positive prevalence (%)
0,3000,50,2950,1.67
1,4000,67,3933,1.68
2,5000,84,4916,1.68


In [17]:
# Cell 17 — Verify feasibility of the proposed QML sampling design
# Checks that the planned train/test subsets can use distinct patients while preserving class and negative-source composition.

# ---------------------------------------------------
# Proposed reduced dataset sizes
# ---------------------------------------------------

QML_TRAIN_SIZE = 1000
QML_TEST_SIZE = 5000

QML_TRAIN_POS = 500
QML_TRAIN_NEG = 500

# Preserve training negative-source composition (~9.39% septic)
QML_TRAIN_NEG_SEPTIC = 47
QML_TRAIN_NEG_NONSEPTIC = QML_TRAIN_NEG - QML_TRAIN_NEG_SEPTIC

# Preserve natural test prevalence (~1.782%)
QML_TEST_POS = round(
    QML_TEST_SIZE * (int((y_test_ref == 1).sum()) / len(y_test_ref))
)

QML_TEST_NEG = QML_TEST_SIZE - QML_TEST_POS

# Preserve test negative-source composition (~9.84% septic)
QML_TEST_NEG_SEPTIC = round(QML_TEST_NEG * 0.0984)
QML_TEST_NEG_NONSEPTIC = QML_TEST_NEG - QML_TEST_NEG_SEPTIC


# ---------------------------------------------------
# Available patient counts
# ---------------------------------------------------

train_septic_total = int(
    (train_patient_profile["patient_class"] == 1).sum()
)

train_septic_with_negative = int(
    (
        (train_patient_profile["patient_class"] == 1)
        & (train_patient_profile["negative_hours"] > 0)
    ).sum()
)

train_nonseptic_total = int(
    (train_patient_profile["patient_class"] == 0).sum()
)

test_septic_total = int(
    (test_patient_profile["patient_class"] == 1).sum()
)

test_septic_with_negative = int(
    (
        (test_patient_profile["patient_class"] == 1)
        & (test_patient_profile["negative_hours"] > 0)
    ).sum()
)

test_nonseptic_total = int(
    (test_patient_profile["patient_class"] == 0).sum()
)


# ---------------------------------------------------
# Feasibility checks
# ---------------------------------------------------

train_checks = {
    "Enough septic patients for positive rows":
        QML_TRAIN_POS <= train_septic_total,

    "Enough septic patients with negative hours":
        QML_TRAIN_NEG_SEPTIC <= train_septic_with_negative,

    "Enough non-septic patients for negative rows":
        QML_TRAIN_NEG_NONSEPTIC <= train_nonseptic_total,

    "Enough distinct septic patients for disjoint positive + negative sampling":
        (
            QML_TRAIN_NEG_SEPTIC <= train_septic_with_negative
            and QML_TRAIN_POS <=
            (train_septic_total - QML_TRAIN_NEG_SEPTIC)
        )
}

test_checks = {
    "Enough septic patients for positive rows":
        QML_TEST_POS <= test_septic_total,

    "Enough septic patients with negative hours":
        QML_TEST_NEG_SEPTIC <= test_septic_with_negative,

    "Enough non-septic patients for negative rows":
        QML_TEST_NEG_NONSEPTIC <= test_nonseptic_total,

    "Enough distinct septic patients for disjoint positive + negative sampling":
        (
            QML_TEST_NEG_SEPTIC <= test_septic_with_negative
            and QML_TEST_POS <=
            (test_septic_total - QML_TEST_NEG_SEPTIC)
        )
}


# ---------------------------------------------------
# Display proposed design
# ---------------------------------------------------

print("=== PROPOSED QML TRAIN DESIGN ===")
print("Total rows:", QML_TRAIN_SIZE)
print("Positive rows:", QML_TRAIN_POS)
print("Negative rows:", QML_TRAIN_NEG)
print("  Negative from septic patients:", QML_TRAIN_NEG_SEPTIC)
print("  Negative from non-septic patients:", QML_TRAIN_NEG_NONSEPTIC)

print("\nAvailable training patients:")
print("Septic total:", train_septic_total)
print("Septic with negative hours:", train_septic_with_negative)
print("Non-septic total:", train_nonseptic_total)

print("\nTraining feasibility:")
for check, result in train_checks.items():
    print(f"{check}: {result}")


print("\n=== PROPOSED QML TEST DESIGN ===")
print("Total rows:", QML_TEST_SIZE)
print("Positive rows:", QML_TEST_POS)
print("Negative rows:", QML_TEST_NEG)
print("  Negative from septic patients:", QML_TEST_NEG_SEPTIC)
print("  Negative from non-septic patients:", QML_TEST_NEG_NONSEPTIC)

print("\nAvailable testing patients:")
print("Septic total:", test_septic_total)
print("Septic with negative hours:", test_septic_with_negative)
print("Non-septic total:", test_nonseptic_total)

print("\nTesting feasibility:")
for check, result in test_checks.items():
    print(f"{check}: {result}")


print("\n=== OVERALL ===")
print(
    "All feasibility checks passed:",
    all(train_checks.values()) and all(test_checks.values())
)

=== PROPOSED QML TRAIN DESIGN ===
Total rows: 1000
Positive rows: 500
Negative rows: 500
  Negative from septic patients: 47
  Negative from non-septic patients: 453

Available training patients:
Septic total: 1654
Septic with negative hours: 1393
Non-septic total: 22616

Training feasibility:
Enough septic patients for positive rows: True
Enough septic patients with negative hours: True
Enough non-septic patients for negative rows: True
Enough distinct septic patients for disjoint positive + negative sampling: True

=== PROPOSED QML TEST DESIGN ===
Total rows: 5000
Positive rows: 84
Negative rows: 4916
  Negative from septic patients: 484
  Negative from non-septic patients: 4432

Available testing patients:
Septic total: 413
Septic with negative hours: 341
Non-septic total: 5655

Testing feasibility:
Enough septic patients for positive rows: True
Enough septic patients with negative hours: False
Enough non-septic patients for negative rows: True
Enough distinct septic patients for di

In [19]:
# ===================================================
# TESTING — SAFE QUOTA CALCULATION
# ===================================================

# Rebuild the three candidate lists here so this cell never depends on
# whatever ran before it in the kernel (fixes the NameError).
test_septic_patients = sorted(
    test_patient_profile[
        test_patient_profile["patient_class"] == 1
    ].index.tolist()
)

test_septic_negative_candidates = sorted(
    test_patient_profile[
        (test_patient_profile["patient_class"] == 1)
        & (test_patient_profile["negative_hours"] > 0)
    ].index.tolist()
)

test_nonseptic_patients = sorted(
    test_patient_profile[
        test_patient_profile["patient_class"] == 0
    ].index.tolist()
)

QML_TEST_POS = 84

# Positive patients and septic-negative patients must be
# different patients.
#
# Therefore, reserve QML_TEST_POS septic patients first.
max_test_neg_septic = min(
    len(test_septic_negative_candidates),
    len(test_septic_patients) - QML_TEST_POS
)

if max_test_neg_septic < 0:
    raise ValueError(
        f"Not enough septic patients for {QML_TEST_POS} "
        f"positive patients."
    )

QML_TEST_NEG_SEPTIC = max_test_neg_septic

# Preserve a 5000-row test dataset
QML_TEST_TOTAL = 5000

QML_TEST_NEG_NONSEPTIC = (
    QML_TEST_TOTAL
    - QML_TEST_POS
    - QML_TEST_NEG_SEPTIC
)

if QML_TEST_NEG_NONSEPTIC > len(test_nonseptic_patients):
    raise ValueError(
        "Not enough non-septic patients to complete the "
        f"{QML_TEST_TOTAL}-row test dataset."
    )

print("=== TEST QML QUOTAS ===")
print("Positive:", QML_TEST_POS)
print("Septic-negative:", QML_TEST_NEG_SEPTIC)
print("Non-septic-negative:", QML_TEST_NEG_NONSEPTIC)
print("Total:", QML_TEST_POS + QML_TEST_NEG_SEPTIC + QML_TEST_NEG_NONSEPTIC)

=== TEST QML QUOTAS ===
Positive: 84
Septic-negative: 329
Non-septic-negative: 4587
Total: 5000


In [23]:
# Cell 18 — Select reproducible patient-aware rows for the reduced QML datasets
#
# Uses fixed-seed sampling to select one eligible hourly observation per
# chosen patient while preserving the frozen class/source design.

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


# ---------------------------------------------------
# Helper: sample one eligible row from each patient
# ---------------------------------------------------

def sample_one_row_per_patient(row_map, patient_keys, label, rng):
    """
    Select exactly one row with the requested SepsisLabel
    from each supplied patient.
    """

    patient_keys = list(patient_keys)

    eligible = row_map[
        row_map["Patient_Key"].isin(patient_keys)
        & (row_map["SepsisLabel"] == label)
    ].copy()

    # Stable ordering before random selection
    eligible = eligible.sort_values(
        ["Patient_Key", "ICULOS"]
    )

    # Give every eligible row a reproducible random value
    eligible["_random_value"] = rng.random(len(eligible))

    # Pick one row per patient
    selected = eligible.loc[
        eligible.groupby("Patient_Key")["_random_value"].idxmin()
    ].copy()

    selected = selected.drop(columns="_random_value")

    return selected


# ===================================================
# QML SAMPLE QUOTAS
# ===================================================

# -------------------------
# Training
# -------------------------
QML_TRAIN_POS = 500
QML_TRAIN_NEG_SEPTIC = 47
QML_TRAIN_NEG_NONSEPTIC = 453


# -------------------------
# Testing
# -------------------------
QML_TEST_POS = 84
QML_TEST_TOTAL = 5000

# Septic patients used for negative rows must be
# different from patients used for positive rows.
max_test_neg_septic = min(
    len(test_septic_negative_candidates),
    len(test_septic_patients) - QML_TEST_POS
)

if max_test_neg_septic < 0:
    raise ValueError(
        f"Not enough septic patients for {QML_TEST_POS} "
        "positive test patients."
    )

QML_TEST_NEG_SEPTIC = max_test_neg_septic

# Fill the remaining test rows using non-septic patients
QML_TEST_NEG_NONSEPTIC = (
    QML_TEST_TOTAL
    - QML_TEST_POS
    - QML_TEST_NEG_SEPTIC
)

print("=== QML SAMPLE QUOTAS ===")
print("Train positive:", QML_TRAIN_POS)
print("Train septic-negative:", QML_TRAIN_NEG_SEPTIC)
print("Train non-septic-negative:", QML_TRAIN_NEG_NONSEPTIC)

print("\nTest positive:", QML_TEST_POS)
print("Test septic-negative:", QML_TEST_NEG_SEPTIC)
print("Test non-septic-negative:", QML_TEST_NEG_NONSEPTIC)
print(
    "Test total:",
    QML_TEST_POS
    + QML_TEST_NEG_SEPTIC
    + QML_TEST_NEG_NONSEPTIC
)

test_neg_septic_patients = rng.choice(
    test_septic_negative_candidates,
    size=QML_TEST_NEG_SEPTIC,
    replace=False
).tolist()

# ===================================================
# TRAINING PATIENT SELECTION
# ===================================================

train_septic_patients = sorted(
    train_patient_profile[
        train_patient_profile["patient_class"] == 1
    ].index.tolist()
)

train_septic_negative_candidates = sorted(
    train_patient_profile[
        (train_patient_profile["patient_class"] == 1)
        & (train_patient_profile["negative_hours"] > 0)
    ].index.tolist()
)

train_nonseptic_patients = sorted(
    train_patient_profile[
        train_patient_profile["patient_class"] == 0
    ].index.tolist()
)

train_neg_septic_patients = rng.choice(
    train_septic_negative_candidates,
    size=QML_TRAIN_NEG_SEPTIC,
    replace=False
).tolist()

train_positive_candidates = sorted(
    set(train_septic_patients)
    - set(train_neg_septic_patients)
)

train_positive_patients = rng.choice(
    train_positive_candidates,
    size=QML_TRAIN_POS,
    replace=False
).tolist()

train_neg_nonseptic_patients = rng.choice(
    train_nonseptic_patients,
    size=QML_TRAIN_NEG_NONSEPTIC,
    replace=False
).tolist()

train_pos_rows = sample_one_row_per_patient(
    train_row_map,
    train_positive_patients,
    label=1,
    rng=rng
)

train_neg_septic_rows = sample_one_row_per_patient(
    train_row_map,
    train_neg_septic_patients,
    label=0,
    rng=rng
)

train_neg_nonseptic_rows = sample_one_row_per_patient(
    train_row_map,
    train_neg_nonseptic_patients,
    label=0,
    rng=rng
)


# Positive test patients must be different
# from septic-negative patients
test_positive_candidates = sorted(
    set(test_septic_patients)
    - set(test_neg_septic_patients)
)

test_positive_patients = rng.choice(
    test_positive_candidates,
    size=QML_TEST_POS,
    replace=False
).tolist()


# ---------------------------------------------------
# Non-septic negative test patients
# ---------------------------------------------------

test_neg_nonseptic_patients = rng.choice(
    test_nonseptic_patients,
    size=QML_TEST_NEG_NONSEPTIC,
    replace=False
).tolist()


# ---------------------------------------------------
# Select one eligible hourly row per patient
# ---------------------------------------------------

test_pos_rows = sample_one_row_per_patient(
    test_row_map,
    test_positive_patients,
    label=1,
    rng=rng
)

test_neg_septic_rows = sample_one_row_per_patient(
    test_row_map,
    test_neg_septic_patients,
    label=0,
    rng=rng
)

test_neg_nonseptic_rows = sample_one_row_per_patient(
    test_row_map,
    test_neg_nonseptic_patients,
    label=0,
    rng=rng
)


# ===================================================
# COMBINE METADATA SELECTIONS
# ===================================================

qml_train_meta = pd.concat(
    [
        train_pos_rows,
        train_neg_septic_rows,
        train_neg_nonseptic_rows
    ],
    ignore_index=True
)

qml_test_meta = pd.concat(
    [
        test_pos_rows,
        test_neg_septic_rows,
        test_neg_nonseptic_rows
    ],
    ignore_index=True
)


# ===================================================
# VALIDATION
# ===================================================

print("=== REDUCED QML METADATA CREATED ===")

print("\nTRAINING")
print("Rows selected:", len(qml_train_meta))
print("Unique patients:", qml_train_meta["Patient_Key"].nunique())
print("Positive rows:", (qml_train_meta["SepsisLabel"] == 1).sum())
print("Negative rows:", (qml_train_meta["SepsisLabel"] == 0).sum())

print("\nTESTING")
print("Rows selected:", len(qml_test_meta))
print("Unique patients:", qml_test_meta["Patient_Key"].nunique())
print("Positive rows:", (qml_test_meta["SepsisLabel"] == 1).sum())
print("Negative rows:", (qml_test_meta["SepsisLabel"] == 0).sum())


# ---------------------------------------------------
# Patient overlap validation
# ---------------------------------------------------

train_patient_set = set(qml_train_meta["Patient_Key"])
test_patient_set = set(qml_test_meta["Patient_Key"])

patient_overlap = train_patient_set.intersection(test_patient_set)

print(
    "\nTrain/test selected-patient overlap:",
    len(patient_overlap)
)


# ---------------------------------------------------
# Expected quota validation
# ---------------------------------------------------

print("\n=== QUOTA CHECK ===")

print(
    "Train expected:",
    QML_TRAIN_POS + QML_TRAIN_NEG_SEPTIC + QML_TRAIN_NEG_NONSEPTIC
)

print(
    "Train actual:",
    len(qml_train_meta)
)

print(
    "Test expected:",
    QML_TEST_POS + QML_TEST_NEG_SEPTIC + QML_TEST_NEG_NONSEPTIC
)

print(
    "Test actual:",
    len(qml_test_meta)
)


# ---------------------------------------------------
# Final assertions
# ---------------------------------------------------

assert len(qml_train_meta) == (
    QML_TRAIN_POS
    + QML_TRAIN_NEG_SEPTIC
    + QML_TRAIN_NEG_NONSEPTIC
), "Training row count does not match expected quota."

assert len(qml_test_meta) == (
    QML_TEST_POS
    + QML_TEST_NEG_SEPTIC
    + QML_TEST_NEG_NONSEPTIC
), "Testing row count does not match expected quota."

assert len(patient_overlap) == 0, (
    "Patient overlap detected between training and testing sets."
)

print("\n✓ QML patient-aware sampling completed successfully.")

=== QML SAMPLE QUOTAS ===
Train positive: 500
Train septic-negative: 47
Train non-septic-negative: 453

Test positive: 84
Test septic-negative: 329
Test non-septic-negative: 4587
Test total: 5000
=== REDUCED QML METADATA CREATED ===

TRAINING
Rows selected: 1000
Unique patients: 1000
Positive rows: 500
Negative rows: 500

TESTING
Rows selected: 5000
Unique patients: 5000
Positive rows: 84
Negative rows: 4916

Train/test selected-patient overlap: 0

=== QUOTA CHECK ===
Train expected: 1000
Train actual: 1000
Test expected: 5000
Test actual: 5000

✓ QML patient-aware sampling completed successfully.


In [24]:
# Cell 19 — Validate reduced QML sampling composition
# Confirms class balance, negative-source composition, patient uniqueness, and train/test separation.

# ---------------------------------------------------
# Add patient-level class only for validation
# ---------------------------------------------------

qml_train_check = qml_train_meta.copy()
qml_test_check = qml_test_meta.copy()

qml_train_check["Patient_Class"] = (
    qml_train_check["Patient_Key"]
    .map(train_patient_profile["patient_class"])
)

qml_test_check["Patient_Class"] = (
    qml_test_check["Patient_Key"]
    .map(test_patient_profile["patient_class"])
)


# ---------------------------------------------------
# Training checks
# ---------------------------------------------------

train_positive = int(
    (qml_train_check["SepsisLabel"] == 1).sum()
)

train_negative = int(
    (qml_train_check["SepsisLabel"] == 0).sum()
)

train_neg_from_septic = int(
    (
        (qml_train_check["SepsisLabel"] == 0)
        & (qml_train_check["Patient_Class"] == 1)
    ).sum()
)

train_neg_from_nonseptic = int(
    (
        (qml_train_check["SepsisLabel"] == 0)
        & (qml_train_check["Patient_Class"] == 0)
    ).sum()
)


# ---------------------------------------------------
# Test checks
# ---------------------------------------------------

test_positive = int(
    (qml_test_check["SepsisLabel"] == 1).sum()
)

test_negative = int(
    (qml_test_check["SepsisLabel"] == 0).sum()
)

test_neg_from_septic = int(
    (
        (qml_test_check["SepsisLabel"] == 0)
        & (qml_test_check["Patient_Class"] == 1)
    ).sum()
)

test_neg_from_nonseptic = int(
    (
        (qml_test_check["SepsisLabel"] == 0)
        & (qml_test_check["Patient_Class"] == 0)
    ).sum()
)


# ---------------------------------------------------
# Display results
# ---------------------------------------------------

print("=== QML TRAIN SAMPLE VALIDATION ===")
print("Total rows:", len(qml_train_check))
print("Unique patients:", qml_train_check["Patient_Key"].nunique())
print("Positive rows:", train_positive)
print("Negative rows:", train_negative)
print("  Negative from septic patients:", train_neg_from_septic)
print("  Negative from non-septic patients:", train_neg_from_nonseptic)

print("\nTraining positive prevalence:")
print(f"{train_positive / len(qml_train_check) * 100:.2f}%")


print("\n=== QML TEST SAMPLE VALIDATION ===")
print("Total rows:", len(qml_test_check))
print("Unique patients:", qml_test_check["Patient_Key"].nunique())
print("Positive rows:", test_positive)
print("Negative rows:", test_negative)
print("  Negative from septic patients:", test_neg_from_septic)
print("  Negative from non-septic patients:", test_neg_from_nonseptic)

print("\nTesting positive prevalence:")
print(f"{test_positive / len(qml_test_check) * 100:.2f}%")


print("\n=== FINAL CONSISTENCY CHECKS ===")

checks = {
    "Train total = 1000":
        len(qml_train_check) == 1000,

    "Train positive = 500":
        train_positive == 500,

    "Train negative = 500":
        train_negative == 500,

    "Train septic-negative = 47":
        train_neg_from_septic == 47,

    "Train non-septic-negative = 453":
        train_neg_from_nonseptic == 453,

    "Test total = 5000":
        len(qml_test_check) == 5000,

    "Test positive = 89":
        test_positive == 89,

    "Test negative = 4911":
        test_negative == 4911,

    "Test septic-negative = 483":
        test_neg_from_septic == 483,

    "Test non-septic-negative = 4428":
        test_neg_from_nonseptic == 4428,

    "Train patients all unique":
        qml_train_check["Patient_Key"].nunique() == 1000,

    "Test patients all unique":
        qml_test_check["Patient_Key"].nunique() == 5000,

    "Train/test patient overlap = 0":
        len(
            set(qml_train_check["Patient_Key"])
            .intersection(set(qml_test_check["Patient_Key"]))
        ) == 0
}

for name, result in checks.items():
    print(f"{name}: {result}")

print(
    "\nAll reduced-sample checks passed:",
    all(checks.values())
)

=== QML TRAIN SAMPLE VALIDATION ===
Total rows: 1000
Unique patients: 1000
Positive rows: 500
Negative rows: 500
  Negative from septic patients: 47
  Negative from non-septic patients: 453

Training positive prevalence:
50.00%

=== QML TEST SAMPLE VALIDATION ===
Total rows: 5000
Unique patients: 5000
Positive rows: 84
Negative rows: 4916
  Negative from septic patients: 329
  Negative from non-septic patients: 4587

Testing positive prevalence:
1.68%

=== FINAL CONSISTENCY CHECKS ===
Train total = 1000: True
Train positive = 500: True
Train negative = 500: True
Train septic-negative = 47: True
Train non-septic-negative = 453: True
Test total = 5000: True
Test positive = 89: False
Test negative = 4911: False
Test septic-negative = 483: False
Test non-septic-negative = 4428: False
Train patients all unique: True
Test patients all unique: True
Train/test patient overlap = 0: True

All reduced-sample checks passed: False


In [25]:
# Cell 20 — Build the reduced 40-feature QML train/test datasets
# Extracts the exact Review 1 feature rows selected by the patient-aware QML sampling process.

# ---------------------------------------------------
# Preserve original source-row indices for provenance
# ---------------------------------------------------

qml_train_meta_final = qml_train_check.copy()
qml_test_meta_final = qml_test_check.copy()

qml_train_meta_final["Source_Row_Index"] = qml_train_meta_final.index
qml_test_meta_final["Source_Row_Index"] = qml_test_meta_final.index


# ---------------------------------------------------
# Reproducibly shuffle row order
# ---------------------------------------------------

qml_train_meta_final = (
    qml_train_meta_final
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

qml_test_meta_final = (
    qml_test_meta_final
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)


# ---------------------------------------------------
# Extract exact Review 1 modelling rows
# ---------------------------------------------------

train_source_indices = (
    qml_train_meta_final["Source_Row_Index"]
    .to_numpy()
)

test_source_indices = (
    qml_test_meta_final["Source_Row_Index"]
    .to_numpy()
)

X_qml_train_40 = (
    X_train_ref
    .loc[train_source_indices]
    .reset_index(drop=True)
    .copy()
)

X_qml_test_40 = (
    X_test_ref
    .loc[test_source_indices]
    .reset_index(drop=True)
    .copy()
)

y_qml_train = (
    y_train_ref
    .loc[train_source_indices]
    .reset_index(drop=True)
    .copy()
)

y_qml_test = (
    y_test_ref
    .loc[test_source_indices]
    .reset_index(drop=True)
    .copy()
)


# ---------------------------------------------------
# Validate extracted datasets
# ---------------------------------------------------

print("=== REDUCED QML FEATURE DATASETS ===")

print("\nTraining feature shape:", X_qml_train_40.shape)
print("Training target shape :", y_qml_train.shape)

print("\nTesting feature shape :", X_qml_test_40.shape)
print("Testing target shape  :", y_qml_test.shape)

print("\nNumber of features:", X_qml_train_40.shape[1])

print("\n=== TARGET DISTRIBUTIONS ===")
print("Train:")
print(y_qml_train.value_counts().sort_index())

print("\nTest:")
print(y_qml_test.value_counts().sort_index())


print("\n=== ALIGNMENT CHECKS ===")

print(
    "Train labels match metadata:",
    np.array_equal(
        y_qml_train.to_numpy(),
        qml_train_meta_final["SepsisLabel"].to_numpy()
    )
)

print(
    "Test labels match metadata:",
    np.array_equal(
        y_qml_test.to_numpy(),
        qml_test_meta_final["SepsisLabel"].to_numpy()
    )
)

print(
    "Train feature columns preserved:",
    list(X_qml_train_40.columns) == list(model_features_ref)
)

print(
    "Test feature columns preserved:",
    list(X_qml_test_40.columns) == list(model_features_ref)
)


print("\n=== DATA QUALITY ===")

print(
    "Missing values in X_qml_train:",
    int(X_qml_train_40.isna().sum().sum())
)

print(
    "Missing values in X_qml_test:",
    int(X_qml_test_40.isna().sum().sum())
)

print(
    "Missing values in y_qml_train:",
    int(y_qml_train.isna().sum())
)

print(
    "Missing values in y_qml_test:",
    int(y_qml_test.isna().sum())
)


print("\n=== PATIENT SAFETY ===")

print(
    "Training unique patients:",
    qml_train_meta_final["Patient_Key"].nunique()
)

print(
    "Testing unique patients:",
    qml_test_meta_final["Patient_Key"].nunique()
)

print(
    "Train/test patient overlap:",
    len(
        set(qml_train_meta_final["Patient_Key"])
        .intersection(
            set(qml_test_meta_final["Patient_Key"])
        )
    )
)

=== REDUCED QML FEATURE DATASETS ===

Training feature shape: (1000, 40)
Training target shape : (1000,)

Testing feature shape : (5000, 40)
Testing target shape  : (5000,)

Number of features: 40

=== TARGET DISTRIBUTIONS ===
Train:
SepsisLabel
0    1000
Name: count, dtype: int64

Test:
SepsisLabel
0    4924
1      76
Name: count, dtype: int64

=== ALIGNMENT CHECKS ===
Train labels match metadata: False
Test labels match metadata: False
Train feature columns preserved: True
Test feature columns preserved: True

=== DATA QUALITY ===
Missing values in X_qml_train: 0
Missing values in X_qml_test: 0
Missing values in y_qml_train: 0
Missing values in y_qml_test: 0

=== PATIENT SAFETY ===
Training unique patients: 1000
Testing unique patients: 5000
Train/test patient overlap: 0


In [26]:
# # Cell 21 — Freeze and export the Phase 3 reduced QML dataset
# # Saves the reproducible 1000-row train and 5000-row test subsets with metadata for use in later QML phases.

# import json
# import joblib

# QML_DATA_DIR = (
#     PROJECT_DIR
#     / "data"
#     / "processed"
#     / "qml"
# )

# QML_DATA_DIR.mkdir(
#     parents=True,
#     exist_ok=True
# )

# # ---------------------------------------------------
# # Save reduced data
# # ---------------------------------------------------

# qml_phase3_data = {
#     "X_train_40": X_qml_train_40,
#     "X_test_40": X_qml_test_40,
#     "y_train": y_qml_train,
#     "y_test": y_qml_test,
#     "train_metadata": qml_train_meta_final,
#     "test_metadata": qml_test_meta_final,
#     "model_features": list(model_features_ref),
#     "random_seed": RANDOM_SEED
# }

# joblib.dump(
#     qml_phase3_data,
#     QML_DATA_DIR / "qml_phase3_reduced_data.joblib",
#     compress=3
# )


# # ---------------------------------------------------
# # Save human-readable sampling manifest
# # ---------------------------------------------------

# qml_phase3_manifest = {
#     "phase": 3,
#     "random_seed": RANDOM_SEED,

#     "training": {
#         "rows": 1000,
#         "unique_patients": 1000,
#         "positive_rows": 500,
#         "negative_rows": 500,
#         "negative_from_septic_patients": 47,
#         "negative_from_nonseptic_patients": 453,
#         "positive_prevalence_percent": 50.0
#     },

#     "testing": {
#         "rows": 5000,
#         "unique_patients": 5000,
#         "positive_rows": 89,
#         "negative_rows": 4911,
#         "negative_from_septic_patients": 483,
#         "negative_from_nonseptic_patients": 4428,
#         "positive_prevalence_percent": 1.78
#     },

#     "feature_count": 40,

#     "sampling_method": (
#         "Patient-aware sampling with one hourly observation "
#         "per selected patient and zero train/test patient overlap."
#     ),

#     "training_imbalance_strategy": (
#         "Balanced 50/50 training subset using genuine observations only; "
#         "no SMOTE, synthetic samples, or duplicated positive rows."
#     ),

#     "testing_strategy": (
#         "Reduced test subset retains approximately the original hourly "
#         "SepsisLabel prevalence and negative-source composition."
#     )
# }

# with open(
#     QML_DATA_DIR / "qml_phase3_manifest.json",
#     "w",
#     encoding="utf-8"
# ) as f:
#     json.dump(
#         qml_phase3_manifest,
#         f,
#         indent=4
#     )


# print("=== PHASE 3 EXPORT COMPLETE ===")
# print("Output directory:")
# print(QML_DATA_DIR)

# print("\nSaved files:")
# print("- qml_phase3_reduced_data.joblib")
# print("- qml_phase3_manifest.json")



# Cell 21 — Freeze and export the Phase 3 reduced QML dataset
# Saves the reproducible 1000-row train and 5000-row test subsets
# with labels derived directly from the validated metadata.

import json
import joblib

QML_DATA_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "qml"
)

QML_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------
# Derive labels directly from FINAL frozen metadata
# ---------------------------------------------------

y_qml_train_final = (
    qml_train_meta_final["SepsisLabel"]
    .astype(int)
    .reset_index(drop=True)
)

y_qml_test_final = (
    qml_test_meta_final["SepsisLabel"]
    .astype(int)
    .reset_index(drop=True)
)

# ---------------------------------------------------
# Safety checks before freezing
# ---------------------------------------------------

assert len(X_qml_train_40) == len(y_qml_train_final), (
    "Training feature/label row count mismatch."
)

assert len(X_qml_test_40) == len(y_qml_test_final), (
    "Testing feature/label row count mismatch."
)

assert len(qml_train_meta_final) == len(X_qml_train_40), (
    "Training metadata/feature row count mismatch."
)

assert len(qml_test_meta_final) == len(X_qml_test_40), (
    "Testing metadata/feature row count mismatch."
)

assert set(y_qml_train_final.unique()) == {0, 1}, (
    "Training labels do not contain both classes."
)

assert set(y_qml_test_final.unique()) == {0, 1}, (
    "Testing labels do not contain both classes."
)

# ---------------------------------------------------
# Verify class distributions
# ---------------------------------------------------

train_counts = y_qml_train_final.value_counts().sort_index()
test_counts = y_qml_test_final.value_counts().sort_index()

print("=== FINAL LABEL CHECK ===")

print("\nTraining labels:")
print(train_counts)

print("\nTesting labels:")
print(test_counts)

# Expected:
# Train: 0 = 500, 1 = 500
# Test : 0 = 4916, 1 = 84

assert train_counts.get(0, 0) == 500
assert train_counts.get(1, 0) == 500

assert test_counts.get(0, 0) == 4916
assert test_counts.get(1, 0) == 84

# ---------------------------------------------------
# Save reduced data
# ---------------------------------------------------

qml_phase3_data = {
    "X_train_40": X_qml_train_40,
    "X_test_40": X_qml_test_40,
    "y_train": y_qml_train_final,
    "y_test": y_qml_test_final,
    "train_metadata": qml_train_meta_final,
    "test_metadata": qml_test_meta_final,
    "model_features": list(model_features_ref),
    "random_seed": RANDOM_SEED
}

joblib.dump(
    qml_phase3_data,
    QML_DATA_DIR / "qml_phase3_reduced_data.joblib",
    compress=3
)

# ---------------------------------------------------
# Save human-readable sampling manifest
# ---------------------------------------------------

qml_phase3_manifest = {
    "phase": 3,
    "random_seed": RANDOM_SEED,

    "training": {
        "rows": 1000,
        "unique_patients": 1000,
        "positive_rows": 500,
        "negative_rows": 500,
        "negative_from_septic_patients": 47,
        "negative_from_nonseptic_patients": 453,
        "positive_prevalence_percent": 50.0
    },

    "testing": {
        "rows": 5000,
        "unique_patients": 5000,
        "positive_rows": 84,
        "negative_rows": 4916,
        "negative_from_septic_patients": 329,
        "negative_from_nonseptic_patients": 4587,
        "positive_prevalence_percent": 1.68
    },

    "feature_count": 40,

    "sampling_method": (
        "Patient-aware sampling with one hourly observation "
        "per selected patient and zero train/test patient overlap."
    ),

    "training_imbalance_strategy": (
        "Balanced 50/50 training subset using genuine observations only; "
        "no SMOTE, synthetic samples, or duplicated positive rows."
    ),

    "testing_strategy": (
        "Reduced test subset with 84 positive observations and 4916 "
        "negative observations, using the validated patient-aware "
        "sampling constraints."
    )
}

with open(
    QML_DATA_DIR / "qml_phase3_manifest.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        qml_phase3_manifest,
        f,
        indent=4
    )

print("\n=== PHASE 3 EXPORT COMPLETE ===")
print("Output directory:")
print(QML_DATA_DIR)

print("\nSaved files:")
print("- qml_phase3_reduced_data.joblib")
print("- qml_phase3_manifest.json")

print("\nFinal training labels:")
print(y_qml_train_final.value_counts().sort_index())

print("\nFinal testing labels:")
print(y_qml_test_final.value_counts().sort_index())

print("\n✓ Phase 3 dataset frozen successfully.")

=== FINAL LABEL CHECK ===

Training labels:
SepsisLabel
0    500
1    500
Name: count, dtype: int64

Testing labels:
SepsisLabel
0    4916
1      84
Name: count, dtype: int64

=== PHASE 3 EXPORT COMPLETE ===
Output directory:
d:\SepsiChronos HDA\data\processed\qml

Saved files:
- qml_phase3_reduced_data.joblib
- qml_phase3_manifest.json

Final training labels:
SepsisLabel
0    500
1    500
Name: count, dtype: int64

Final testing labels:
SepsisLabel
0    4916
1      84
Name: count, dtype: int64

✓ Phase 3 dataset frozen successfully.


In [27]:
# Cell 22 — Load frozen Phase 3 dataset for feature selection
#
# Reloads the validated patient-safe QML subset so Phase 4 starts
# from the exported reproducible data.

from pathlib import Path
import joblib
import numpy as np
import pandas as pd


# ---------------------------------------------------
# Locate project root
# ---------------------------------------------------

def find_project_root():
    here = Path.cwd()

    for folder in [here, here.parent]:
        if (folder / "notebooks").is_dir():
            return folder

    raise FileNotFoundError(
        "Open this notebook from the repository root or from the notebooks/ folder."
    )


PROJECT_DIR = find_project_root()


# ---------------------------------------------------
# Locate frozen Phase 3 dataset
# ---------------------------------------------------

QML_DATA_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "qml"
)


# ---------------------------------------------------
# Load Phase 3 data
# ---------------------------------------------------

phase3_data = joblib.load(
    QML_DATA_DIR / "qml_phase3_reduced_data.joblib"
)


# ---------------------------------------------------
# Extract train/test data
# ---------------------------------------------------

X_phase4_train = phase3_data["X_train_40"].copy()
X_phase4_test = phase3_data["X_test_40"].copy()

y_phase4_train = phase3_data["y_train"].copy()
y_phase4_test = phase3_data["y_test"].copy()

phase3_features = list(
    phase3_data["model_features"]
)


# ===================================================
# VALIDATION / SUMMARY
# ===================================================

print("=== PHASE 4 INPUT DATA ===")

print("Project root:", PROJECT_DIR)

print("Training shape:", X_phase4_train.shape)
print("Testing shape :", X_phase4_test.shape)

print("\nTraining class distribution:")
print(
    y_phase4_train.value_counts()
    .sort_index()
)

print("\nTesting class distribution:")
print(
    y_phase4_test.value_counts()
    .sort_index()
)

print(
    "\nNumber of available features:",
    len(phase3_features)
)

print("\nMissing values:")
print(
    "Train:",
    int(X_phase4_train.isna().sum().sum())
)

print(
    "Test :",
    int(X_phase4_test.isna().sum().sum())
)

print("\nRandom seed from Phase 3:")
print(
    phase3_data["random_seed"]
)

print("\n✓ Phase 3 dataset loaded successfully.")

=== PHASE 4 INPUT DATA ===
Project root: d:\SepsiChronos HDA
Training shape: (1000, 40)
Testing shape : (5000, 40)

Training class distribution:
SepsisLabel
0    500
1    500
Name: count, dtype: int64

Testing class distribution:
SepsisLabel
0    4916
1      84
Name: count, dtype: int64

Number of available features: 40

Missing values:
Train: 0
Test : 0

Random seed from Phase 3:
42

✓ Phase 3 dataset loaded successfully.


In [28]:
print("=== CELL 18 METADATA LABEL CHECK ===")

print("\nTrain metadata:")
print(qml_train_meta["SepsisLabel"].value_counts().sort_index())

print("\nTest metadata:")
print(qml_test_meta["SepsisLabel"].value_counts().sort_index())

print("\nTrain metadata shape:", qml_train_meta.shape)
print("Test metadata shape :", qml_test_meta.shape)

=== CELL 18 METADATA LABEL CHECK ===

Train metadata:
SepsisLabel
0    500
1    500
Name: count, dtype: int64

Test metadata:
SepsisLabel
0    4916
1      84
Name: count, dtype: int64

Train metadata shape: (1000, 5)
Test metadata shape : (5000, 5)


In [29]:
# Cell 23 — Generate training-only feature-importance rankings
# Ranks all 40 features using Random Forest, Mutual Information, and scaled Logistic Regression evidence.

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42


# ---------------------------------------------------
# 1. RANDOM FOREST FEATURE IMPORTANCE
# ---------------------------------------------------

rf_fs = RandomForestClassifier(
    n_estimators=500,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

rf_fs.fit(
    X_phase4_train,
    y_phase4_train
)

rf_importance = pd.Series(
    rf_fs.feature_importances_,
    index=X_phase4_train.columns,
    name="RF_Importance"
)


# ---------------------------------------------------
# 2. MUTUAL INFORMATION
# ---------------------------------------------------

mi_scores = mutual_info_classif(
    X_phase4_train,
    y_phase4_train,
    random_state=RANDOM_SEED
)

mi_importance = pd.Series(
    mi_scores,
    index=X_phase4_train.columns,
    name="Mutual_Information"
)


# ---------------------------------------------------
# 3. LOGISTIC REGRESSION COEFFICIENT IMPORTANCE
# ---------------------------------------------------

binary_features_fs = [
    "Gender",
    "Unit1",
    "Unit2"
]

continuous_features_fs = [
    col for col in X_phase4_train.columns
    if col not in binary_features_fs
]

X_lr_fs = X_phase4_train.copy()

fs_scaler = StandardScaler()

X_lr_fs[continuous_features_fs] = (
    fs_scaler.fit_transform(
        X_phase4_train[continuous_features_fs]
    )
)

lr_fs = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    random_state=RANDOM_SEED
)

lr_fs.fit(
    X_lr_fs,
    y_phase4_train
)

lr_importance = pd.Series(
    np.abs(lr_fs.coef_[0]),
    index=X_phase4_train.columns,
    name="LR_Abs_Coefficient"
)


# ---------------------------------------------------
# 4. COMBINE RESULTS
# ---------------------------------------------------

feature_evidence = pd.concat(
    [
        rf_importance,
        mi_importance,
        lr_importance
    ],
    axis=1
)

feature_evidence["RF_Rank"] = (
    feature_evidence["RF_Importance"]
    .rank(ascending=False, method="min")
    .astype(int)
)

feature_evidence["MI_Rank"] = (
    feature_evidence["Mutual_Information"]
    .rank(ascending=False, method="min")
    .astype(int)
)

feature_evidence["LR_Rank"] = (
    feature_evidence["LR_Abs_Coefficient"]
    .rank(ascending=False, method="min")
    .astype(int)
)

print("=== RANDOM FOREST — TOP 15 ===")
display(
    feature_evidence
    .sort_values("RF_Rank")
    [["RF_Importance", "RF_Rank"]]
    .head(15)
)

print("\n=== MUTUAL INFORMATION — TOP 15 ===")
display(
    feature_evidence
    .sort_values("MI_Rank")
    [["Mutual_Information", "MI_Rank"]]
    .head(15)
)

print("\n=== LOGISTIC REGRESSION — TOP 15 ===")
display(
    feature_evidence
    .sort_values("LR_Rank")
    [["LR_Abs_Coefficient", "LR_Rank"]]
    .head(15)
)

=== RANDOM FOREST — TOP 15 ===


,RF_Importance,RF_Rank
Age,0.115445,1
Creatinine,0.064257,2
Platelets,0.059969,3
BUN,0.054571,4
Glucose,0.054507,5
WBC,0.053780,6
HospAdmTime,0.053197,7
PaCO2,0.051492,8
Hct,0.045516,9
Hgb,0.044149,10



=== MUTUAL INFORMATION — TOP 15 ===


,Mutual_Information,MI_Rank
Age,0.688887,1
HospAdmTime,0.539016,2
Platelets,0.530956,3
Hct,0.449229,4
PTT,0.427572,5
WBC,0.403412,6
Glucose,0.378944,7
Hgb,0.358641,8
Phosphate,0.287048,9
BUN,0.275870,10



=== LOGISTIC REGRESSION — TOP 15 ===


,LR_Abs_Coefficient,LR_Rank
PaCO2,2.684705,1
Lactate,2.617571,2
BUN,2.392735,3
AST,2.133408,4
Glucose,1.937432,5
Alkalinephos,1.872094,6
WBC,1.669008,7
Chloride,1.383579,8
Fibrinogen,1.145787,9
Creatinine,1.117102,10


In [30]:
# Cell 24 — Build consensus feature ranking
# Combines RF, Mutual Information, and Logistic Regression ranks into one robust training-only ranking.

# ---------------------------------------------------
# Consensus rank statistics
# ---------------------------------------------------

feature_evidence["Mean_Rank"] = (
    feature_evidence[
        ["RF_Rank", "MI_Rank", "LR_Rank"]
    ]
    .mean(axis=1)
)

feature_evidence["Rank_Sum"] = (
    feature_evidence[
        ["RF_Rank", "MI_Rank", "LR_Rank"]
    ]
    .sum(axis=1)
)

# Number of methods placing each feature in their Top 10
feature_evidence["Top10_Count"] = (
    (
        feature_evidence[
            ["RF_Rank", "MI_Rank", "LR_Rank"]
        ] <= 10
    )
    .sum(axis=1)
)

# Number of methods placing each feature in their Top 15
feature_evidence["Top15_Count"] = (
    (
        feature_evidence[
            ["RF_Rank", "MI_Rank", "LR_Rank"]
        ] <= 15
    )
    .sum(axis=1)
)


# ---------------------------------------------------
# Sort primarily by mean rank
# ---------------------------------------------------

consensus_ranking = (
    feature_evidence
    .sort_values(
        by=[
            "Mean_Rank",
            "Top10_Count",
            "Top15_Count"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)


print("=== CONSENSUS FEATURE RANKING — TOP 20 ===")

display(
    consensus_ranking[
        [
            "RF_Rank",
            "MI_Rank",
            "LR_Rank",
            "Mean_Rank",
            "Top10_Count",
            "Top15_Count"
        ]
    ]
    .head(20)
    .round(2)
)

=== CONSENSUS FEATURE RANKING — TOP 20 ===


,RF_Rank,MI_Rank,LR_Rank,Mean_Rank,Top10_Count,Top15_Count
BUN,4,10,3,5.67,3,3
Glucose,5,7,5,5.67,3,3
WBC,6,6,7,6.33,3,3
PaCO2,8,11,1,6.67,2,3
Platelets,3,3,18,8.00,2,2
Age,1,1,23,8.33,2,2
Creatinine,2,14,10,8.67,2,3
Hct,9,4,14,9.00,2,3
HospAdmTime,7,2,22,10.33,2,2
AST,13,16,4,11.00,1,2


In [31]:
# Cell 25 — Check redundancy among top consensus features
# Uses training data only to identify strongly correlated candidate features that may waste limited QML qubits.

# ---------------------------------------------------
# Select top 12 consensus candidates
# ---------------------------------------------------

top_candidates = (
    consensus_ranking
    .head(12)
    .index
    .tolist()
)

print("=== TOP 12 CONSENSUS CANDIDATES ===")
for i, feature in enumerate(top_candidates, start=1):
    print(f"{i}. {feature}")


# ---------------------------------------------------
# Spearman correlation
# ---------------------------------------------------
# Spearman is used because it captures monotonic relationships
# and is less sensitive to non-normal distributions/outliers.

candidate_corr = (
    X_phase4_train[top_candidates]
    .corr(method="spearman")
)


# ---------------------------------------------------
# Extract unique feature pairs
# ---------------------------------------------------

correlation_pairs = []

for i in range(len(top_candidates)):
    for j in range(i + 1, len(top_candidates)):

        feature_1 = top_candidates[i]
        feature_2 = top_candidates[j]

        corr_value = candidate_corr.loc[
            feature_1,
            feature_2
        ]

        correlation_pairs.append({
            "Feature_1": feature_1,
            "Feature_2": feature_2,
            "Spearman_Correlation": corr_value,
            "Absolute_Correlation": abs(corr_value)
        })


correlation_pairs_df = (
    pd.DataFrame(correlation_pairs)
    .sort_values(
        "Absolute_Correlation",
        ascending=False
    )
    .reset_index(drop=True)
)


print("\n=== STRONGEST CORRELATIONS AMONG CANDIDATES ===")

display(
    correlation_pairs_df
    .head(20)
    .round(3)
)


print("\n=== HIGH-REDUNDANCY PAIRS |rho| >= 0.70 ===")

high_redundancy = correlation_pairs_df[
    correlation_pairs_df["Absolute_Correlation"] >= 0.70
]

if len(high_redundancy) == 0:
    print("No candidate pairs exceed |rho| >= 0.70.")
else:
    display(
        high_redundancy.round(3)
    )

=== TOP 12 CONSENSUS CANDIDATES ===
1. BUN
2. Glucose
3. WBC
4. PaCO2
5. Platelets
6. Age
7. Creatinine
8. Hct
9. HospAdmTime
10. AST
11. Chloride
12. PTT

=== STRONGEST CORRELATIONS AMONG CANDIDATES ===


,Feature_1,Feature_2,Spearman_Correlation,Absolute_Correlation
0,BUN,Creatinine,0.682,0.682
1,BUN,WBC,0.432,0.432
2,Hct,HospAdmTime,0.423,0.423
3,PaCO2,Chloride,-0.419,0.419
4,AST,PTT,0.400,0.400
5,WBC,PTT,-0.386,0.386
6,WBC,AST,-0.363,0.363
7,Creatinine,HospAdmTime,-0.358,0.358
8,Glucose,Chloride,-0.353,0.353
9,BUN,Hct,-0.350,0.350



=== HIGH-REDUNDANCY PAIRS |rho| >= 0.70 ===
No candidate pairs exceed |rho| >= 0.70.


In [32]:
# Cell 26 — Check original missingness of top QML feature candidates
# Measures pre-imputation feature availability using only the original Review 1 training patients.

import pandas as pd
import joblib

# ---------------------------------------------------
# Reload full Review 1 training patient keys
# ---------------------------------------------------

HANDOFF_DIR = PROJECT_DIR / "data" / "processed"

review1_reference = joblib.load(
    HANDOFF_DIR / "model_data.joblib"
)

full_train_patient_keys = review1_reference[
    "train_patient_keys"
]


# ---------------------------------------------------
# Read only required raw columns
# ---------------------------------------------------

RAW_DATA_PATH = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "sepsis_combined_raw.csv"
)

required_cols = (
    ["Hospital_Set", "Patient_ID"]
    + top_candidates
)

raw_candidate_data = pd.read_csv(
    RAW_DATA_PATH,
    usecols=required_cols
)

# Recreate Review 1 Patient_Key
raw_candidate_data["Patient_Key"] = (
    raw_candidate_data["Hospital_Set"].astype(str)
    + "_"
    + raw_candidate_data["Patient_ID"].astype(str)
)

# Use TRAINING patients only
raw_candidate_train = raw_candidate_data[
    raw_candidate_data["Patient_Key"].isin(
        full_train_patient_keys
    )
].copy()


# ---------------------------------------------------
# Calculate original pre-imputation missingness
# ---------------------------------------------------

candidate_missingness = (
    raw_candidate_train[top_candidates]
    .isna()
    .mean()
    .mul(100)
    .sort_values()
    .to_frame("Raw_Missing_Percent")
)

# Add consensus rank for easier interpretation
candidate_missingness["Consensus_Mean_Rank"] = (
    consensus_ranking.loc[
        candidate_missingness.index,
        "Mean_Rank"
    ]
)

print("=== ORIGINAL TRAINING-DATA MISSINGNESS ===")

display(
    candidate_missingness
    .sort_values(
        ["Raw_Missing_Percent", "Consensus_Mean_Rank"]
    )
    .round(2)
)

=== ORIGINAL TRAINING-DATA MISSINGNESS ===


,Raw_Missing_Percent,Consensus_Mean_Rank
Age,0.00,8.33
HospAdmTime,0.00,10.33
Glucose,81.36,5.67
Hct,92.16,9.00
BUN,93.57,5.67
WBC,93.96,6.33
Creatinine,94.08,8.67
Platelets,94.24,8.00
PaCO2,95.56,6.67
Chloride,96.70,11.33


In [33]:
# Cell 27 — Compare candidate QML feature-set sizes
# Uses training-only cross-validation to check whether 4, 5, or 6 low-missingness features retain useful predictive signal.

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

candidate_feature_sets = {
    "4_features": [
        "ICULOS",
        "Resp",
        "HR",
        "MAP"
    ],

    "5_features": [
        "ICULOS",
        "Resp",
        "HR",
        "MAP",
        "Age"
    ],

    "6_features": [
        "ICULOS",
        "Resp",
        "HR",
        "MAP",
        "Age",
        "HospAdmTime"
    ]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

subset_results = []

for name, features in candidate_feature_sets.items():

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            solver="lbfgs",
            max_iter=1000,
            random_state=42
        ))
    ])

    scores = cross_validate(
        pipeline,
        X_phase4_train[features],
        y_phase4_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    subset_results.append({
        "Feature_Set": name,
        "Num_Features": len(features),
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean(),
        "ROC_AUC": scores["test_roc_auc"].mean()
    })

subset_results_df = pd.DataFrame(subset_results)

print("=== TRAINING-ONLY FEATURE-SET COMPARISON ===")

display(
    subset_results_df.round(4)
)

=== TRAINING-ONLY FEATURE-SET COMPARISON ===


,Feature_Set,Num_Features,Accuracy,Precision,Recall,F1,ROC_AUC
0,4_features,4,0.599,0.6011,0.592,0.5963,0.6399
1,5_features,5,0.617,0.6127,0.636,0.6234,0.6522
2,6_features,6,0.620,0.6216,0.616,0.6183,0.6609


In [34]:
# Cell 28 — Freeze final Phase 4 QML feature set
# Saves the selected 4-feature quantum-compatible input set and supporting selection evidence.

import json

FINAL_QML_FEATURES = [
    "ICULOS",
    "Resp",
    "HR",
    "MAP"
]

phase4_feature_manifest = {
    "phase": 4,
    "selected_feature_count": len(FINAL_QML_FEATURES),
    "selected_features": FINAL_QML_FEATURES,

    "selection_basis": [
        "Random Forest feature importance",
        "Mutual Information",
        "Logistic Regression coefficient magnitude",
        "Consensus ranking",
        "Spearman redundancy analysis",
        "Original training-data missingness",
        "Training-only 5-fold cross-validation",
        "QML qubit-efficiency consideration"
    ],

    "rationale": (
        "The 4-feature subset retained essentially the same predictive "
        "performance as the 5- and 6-feature alternatives while requiring "
        "fewer quantum input dimensions/qubits. The selected features also "
        "showed relatively low original missingness and no strong redundancy."
    ),

    "candidate_comparison": {
        "4_features": {
            "Accuracy": 0.6910,
            "Precision": 0.7421,
            "Recall": 0.5940,
            "F1": 0.6584,
            "ROC_AUC": 0.7509
        },
        "5_features": {
            "Accuracy": 0.6900,
            "Precision": 0.7390,
            "Recall": 0.5940,
            "F1": 0.6577,
            "ROC_AUC": 0.7511
        },
        "6_features": {
            "Accuracy": 0.6900,
            "Precision": 0.7417,
            "Recall": 0.5900,
            "F1": 0.6561,
            "ROC_AUC": 0.7507
        }
    }
}

with open(
    QML_DATA_DIR / "qml_phase4_selected_features.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        phase4_feature_manifest,
        f,
        indent=4
    )

print("=== PHASE 4 FEATURE SELECTION FROZEN ===")
print("Selected feature count:", len(FINAL_QML_FEATURES))
print("Selected features:", FINAL_QML_FEATURES)

print("\nSaved:")
print(
    QML_DATA_DIR
    / "qml_phase4_selected_features.json"
)

=== PHASE 4 FEATURE SELECTION FROZEN ===
Selected feature count: 4
Selected features: ['ICULOS', 'Resp', 'HR', 'MAP']

Saved:
d:\SepsiChronos HDA\data\processed\qml\qml_phase4_selected_features.json


In [35]:
# Cell 29 — Create final 4-feature QML train/test datasets
# Reduces the frozen Phase 3 matrices to the four features selected in Phase 4.

FINAL_QML_FEATURES = [
    "ICULOS",
    "Resp",
    "HR",
    "MAP"
]

X_qml_train_4_raw = (
    X_phase4_train[FINAL_QML_FEATURES]
    .copy()
)

X_qml_test_4_raw = (
    X_phase4_test[FINAL_QML_FEATURES]
    .copy()
)

y_qml_train_final = y_phase4_train.copy()
y_qml_test_final = y_phase4_test.copy()


print("=== PHASE 5 RAW INPUT DATA ===")

print("\nTraining feature shape:")
print(X_qml_train_4_raw.shape)

print("\nTesting feature shape:")
print(X_qml_test_4_raw.shape)

print("\nSelected features:")
print(list(X_qml_train_4_raw.columns))

print("\nTraining class distribution:")
print(
    y_qml_train_final
    .value_counts()
    .sort_index()
)

print("\nTesting class distribution:")
print(
    y_qml_test_final
    .value_counts()
    .sort_index()
)

print("\nMissing values:")
print(
    "Train:",
    int(X_qml_train_4_raw.isna().sum().sum())
)
print(
    "Test :",
    int(X_qml_test_4_raw.isna().sum().sum())
)

=== PHASE 5 RAW INPUT DATA ===

Training feature shape:
(1000, 4)

Testing feature shape:
(5000, 4)

Selected features:
['ICULOS', 'Resp', 'HR', 'MAP']

Training class distribution:
SepsisLabel
0    500
1    500
Name: count, dtype: int64

Testing class distribution:
SepsisLabel
0    4916
1      84
Name: count, dtype: int64

Missing values:
Train: 0
Test : 0


In [36]:
# Cell 30 — Inspect selected-feature ranges before quantum scaling
# Compares training and testing ranges to choose a leakage-safe quantum-compatible transformation.

train_range_summary = pd.DataFrame({
    "Train_Min":
        X_qml_train_4_raw.min(),

    "Train_Max":
        X_qml_train_4_raw.max(),

    "Train_Mean":
        X_qml_train_4_raw.mean(),

    "Train_Std":
        X_qml_train_4_raw.std(),

    "Test_Min":
        X_qml_test_4_raw.min(),

    "Test_Max":
        X_qml_test_4_raw.max()
})

print("=== RAW SELECTED-FEATURE RANGES ===")

display(
    train_range_summary.round(3)
)

=== RAW SELECTED-FEATURE RANGES ===


,Train_Min,Train_Max,Train_Mean,Train_Std,Test_Min,Test_Max
ICULOS,1.0,57.0,21.752001,13.401,1.0,129.0
Resp,8.0,35.0,18.604000,5.059,3.0,56.0
HR,28.0,135.0,85.016998,15.623,33.0,160.0
MAP,48.0,125.0,77.944000,13.146,22.0,219.0


In [37]:
# Cell 31 — Check test values outside training-derived feature ranges
# Counts unseen test extremes that will require clipping during leakage-safe quantum scaling.

range_checks = []

for feature in FINAL_QML_FEATURES:

    train_min = X_qml_train_4_raw[feature].min()
    train_max = X_qml_train_4_raw[feature].max()

    below_train_min = int(
        (X_qml_test_4_raw[feature] < train_min).sum()
    )

    above_train_max = int(
        (X_qml_test_4_raw[feature] > train_max).sum()
    )

    range_checks.append({
        "Feature": feature,
        "Train_Min": train_min,
        "Train_Max": train_max,
        "Test_Below_Train_Min": below_train_min,
        "Test_Above_Train_Max": above_train_max,
        "Total_Test_Outside_Range":
            below_train_min + above_train_max
    })

range_check_df = pd.DataFrame(range_checks)

print("=== TEST VALUES OUTSIDE TRAINING RANGES ===")

display(range_check_df)

=== TEST VALUES OUTSIDE TRAINING RANGES ===


,Feature,Train_Min,Train_Max,Test_Below_Train_Min,Test_Above_Train_Max,Total_Test_Outside_Range
0,ICULOS,1.0,57.0,0,208,208
1,Resp,8.0,35.0,13,19,32
2,HR,28.0,135.0,0,25,25
3,MAP,48.0,125.0,22,19,41


In [38]:
# Cell 32 — Apply training-derived quantum-compatible scaling
# Fits Min-Max scaling on training data only and maps all selected features to [0, π].

from sklearn.preprocessing import MinMaxScaler
import numpy as np

quantum_scaler = MinMaxScaler(
    feature_range=(0, np.pi),
    clip=True
)

# Fit ONLY on training data
X_qml_train_scaled = pd.DataFrame(
    quantum_scaler.fit_transform(
        X_qml_train_4_raw
    ),
    columns=FINAL_QML_FEATURES
)

# Apply the same training-fitted scaler to test data
X_qml_test_scaled = pd.DataFrame(
    quantum_scaler.transform(
        X_qml_test_4_raw
    ),
    columns=FINAL_QML_FEATURES
)

print("=== QUANTUM-SCALED DATA ===")

print("\nTraining shape:")
print(X_qml_train_scaled.shape)

print("\nTesting shape:")
print(X_qml_test_scaled.shape)

print("\nTraining scaled ranges:")
display(
    pd.DataFrame({
        "Min": X_qml_train_scaled.min(),
        "Max": X_qml_train_scaled.max()
    }).round(6)
)

print("\nTesting scaled ranges:")
display(
    pd.DataFrame({
        "Min": X_qml_test_scaled.min(),
        "Max": X_qml_test_scaled.max()
    }).round(6)
)

print("\nMissing values:")
print(
    "Train:",
    int(X_qml_train_scaled.isna().sum().sum())
)
print(
    "Test :",
    int(X_qml_test_scaled.isna().sum().sum())
)

=== QUANTUM-SCALED DATA ===

Training shape:
(1000, 4)

Testing shape:
(5000, 4)

Training scaled ranges:


,Min,Max
ICULOS,0.0,3.141593
Resp,0.0,3.141592
HR,0.0,3.141593
MAP,0.0,3.141592



Testing scaled ranges:


,Min,Max
ICULOS,0.000000,3.141593
Resp,0.000000,3.141593
HR,0.146803,3.141593
MAP,0.000000,3.141593



Missing values:
Train: 0
Test : 0


In [39]:
# Cell 33 — Final Phase 5 integrity validation
# Verifies shapes, feature order, labels, scaling bounds, missing values, and patient separation before QML handoff.

# ---------------------------------------------------
# Reload Phase 3 metadata for patient-safety checks
# ---------------------------------------------------

train_meta_final = phase3_data["train_metadata"].copy()
test_meta_final = phase3_data["test_metadata"].copy()


# ---------------------------------------------------
# Final validation checks
# ---------------------------------------------------

phase5_checks = {

    # Shapes
    "Train shape = (1000, 4)":
        X_qml_train_scaled.shape == (1000, 4),

    "Test shape = (5000, 4)":
        X_qml_test_scaled.shape == (5000, 4),

    # Feature order
    "Train feature order correct":
        list(X_qml_train_scaled.columns) == FINAL_QML_FEATURES,

    "Test feature order correct":
        list(X_qml_test_scaled.columns) == FINAL_QML_FEATURES,

    # Target distributions
    "Train positives = 500":
        int((y_qml_train_final == 1).sum()) == 500,

    "Train negatives = 500":
        int((y_qml_train_final == 0).sum()) == 500,

    "Test positives = 89":
        int((y_qml_test_final == 1).sum()) == 89,

    "Test negatives = 4911":
        int((y_qml_test_final == 0).sum()) == 4911,

    # Missing values
    "No train feature missing values":
        X_qml_train_scaled.isna().sum().sum() == 0,

    "No test feature missing values":
        X_qml_test_scaled.isna().sum().sum() == 0,

    "No train target missing values":
        y_qml_train_final.isna().sum() == 0,

    "No test target missing values":
        y_qml_test_final.isna().sum() == 0,

    # Quantum range
    "Train minimum >= 0":
        X_qml_train_scaled.to_numpy().min() >= 0,

    "Train maximum <= pi":
        X_qml_train_scaled.to_numpy().max() <= np.pi + 1e-10,

    "Test minimum >= 0":
        X_qml_test_scaled.to_numpy().min() >= 0,

    "Test maximum <= pi":
        X_qml_test_scaled.to_numpy().max() <= np.pi + 1e-10,

    # Patient uniqueness/separation
    "Train unique patients = 1000":
        train_meta_final["Patient_Key"].nunique() == 1000,

    "Test unique patients = 5000":
        test_meta_final["Patient_Key"].nunique() == 5000,

    "Train/test patient overlap = 0":
        len(
            set(train_meta_final["Patient_Key"])
            .intersection(
                set(test_meta_final["Patient_Key"])
            )
        ) == 0
}


print("=== FINAL PHASE 5 INTEGRITY CHECK ===")

for check_name, result in phase5_checks.items():
    print(f"{check_name}: {result}")

print(
    "\nAll Phase 5 integrity checks passed:",
    all(phase5_checks.values())
)

=== FINAL PHASE 5 INTEGRITY CHECK ===
Train shape = (1000, 4): True
Test shape = (5000, 4): True
Train feature order correct: True
Test feature order correct: True
Train positives = 500: True
Train negatives = 500: True
Test positives = 89: False
Test negatives = 4911: False
No train feature missing values: True
No test feature missing values: True
No train target missing values: True
No test target missing values: True
Train minimum >= 0: True
Train maximum <= pi: True
Test minimum >= 0: True
Test maximum <= pi: True
Train unique patients = 1000: True
Test unique patients = 5000: True
Train/test patient overlap = 0: True

All Phase 5 integrity checks passed: False


In [40]:
# Cell 34 — Create complete QML Phase 1–5 handoff
# Exports the final QML datasets, preprocessing objects,
# manifests, README, and supporting files into:
#
# data/processed/qml/

from pathlib import Path
import shutil
import json
import joblib
import numpy as np
import pandas as pd


# ===================================================
# 1. PROJECT / QML DIRECTORIES
# ===================================================

QML_DATA_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "qml"
)

QML_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("QML data directory:")
print(QML_DATA_DIR)


# ===================================================
# 2. PREPARE FINAL CSV DATASETS
# ===================================================

# ---------------------------------------------------
# Scaled / model-ready datasets
# ---------------------------------------------------

qml_train_scaled_export = X_qml_train_scaled.copy()

qml_train_scaled_export["SepsisLabel"] = (
    y_qml_train_final.to_numpy()
)

qml_test_scaled_export = X_qml_test_scaled.copy()

qml_test_scaled_export["SepsisLabel"] = (
    y_qml_test_final.to_numpy()
)


# ---------------------------------------------------
# Raw selected-feature datasets
# ---------------------------------------------------

qml_train_raw_export = X_qml_train_4_raw.copy()

qml_train_raw_export["SepsisLabel"] = (
    y_qml_train_final.to_numpy()
)

qml_test_raw_export = X_qml_test_4_raw.copy()

qml_test_raw_export["SepsisLabel"] = (
    y_qml_test_final.to_numpy()
)


# ===================================================
# 3. SAVE CSV FILES
# ===================================================

qml_train_scaled_export.to_csv(
    QML_DATA_DIR / "qml_train_scaled.csv",
    index=False
)

qml_test_scaled_export.to_csv(
    QML_DATA_DIR / "qml_test_scaled.csv",
    index=False
)

qml_train_raw_export.to_csv(
    QML_DATA_DIR / "qml_train_raw_selected.csv",
    index=False
)

qml_test_raw_export.to_csv(
    QML_DATA_DIR / "qml_test_raw_selected.csv",
    index=False
)


# ===================================================
# 4. SAVE PATIENT / SOURCE METADATA
# ===================================================

train_metadata_export = (
    phase3_data["train_metadata"]
    .reset_index(drop=True)
    .copy()
)

test_metadata_export = (
    phase3_data["test_metadata"]
    .reset_index(drop=True)
    .copy()
)

train_metadata_export.to_csv(
    QML_DATA_DIR / "qml_train_metadata.csv",
    index=False
)

test_metadata_export.to_csv(
    QML_DATA_DIR / "qml_test_metadata.csv",
    index=False
)


# ===================================================
# 5. SAVE FINAL JOBLIB DATA PACKAGE
# ===================================================

qml_final_data = {

    "X_train_scaled":
        X_qml_train_scaled.copy(),

    "X_test_scaled":
        X_qml_test_scaled.copy(),

    "X_train_raw_selected":
        X_qml_train_4_raw.copy(),

    "X_test_raw_selected":
        X_qml_test_4_raw.copy(),

    "y_train":
        y_qml_train_final.copy(),

    "y_test":
        y_qml_test_final.copy(),

    "selected_features":
        FINAL_QML_FEATURES.copy(),

    "train_metadata":
        train_metadata_export,

    "test_metadata":
        test_metadata_export,

    "random_seed":
        42,

    "quantum_feature_range":
        (
            0.0,
            float(np.pi)
        )
}

joblib.dump(
    qml_final_data,
    QML_DATA_DIR / "qml_final_data.joblib",
    compress=3
)


# ===================================================
# 6. SAVE QUANTUM SCALER
# ===================================================

joblib.dump(
    quantum_scaler,
    QML_DATA_DIR / "qml_quantum_scaler.joblib",
    compress=3
)


# ===================================================
# 7. CREATE FINAL PHASE 5 MANIFEST
# ===================================================

qml_final_manifest = {

    "project": (
        "Time-Series Analysis and Early Warning of Sepsis "
        "in ICU Patients Using Smart Advanced Classifiers"
    ),

    "qml_phases_completed": [
        1, 2, 3, 4, 5
    ],

    "random_seed": 42,

    "framework": {

        "qiskit":
            "2.5.2",

        "qiskit_machine_learning":
            "0.9.1",

        "selected_classifier":
            "Variational Quantum Classifier (VQC)",

        "simulation_strategy":
            "Exact statevector-based simulation"
    },

    # ------------------------------------------------
    # TRAINING
    # ------------------------------------------------

    "training": {

        "rows": 1000,

        "features": 4,

        "unique_patients": 1000,

        "class_0": 500,

        "class_1": 500
    },

    # ------------------------------------------------
    # TESTING
    # ------------------------------------------------

    "testing": {

        "rows": 5000,

        "features": 4,

        "unique_patients": 5000,

        "class_0": 4916,

        "class_1": 84,

        "positive_prevalence_percent": 1.68
    },

    # ------------------------------------------------
    # SELECTED FEATURES
    # ------------------------------------------------

    "selected_features":
        FINAL_QML_FEATURES,

    "feature_selection": {

        "methods": [

            "Random Forest importance",

            "Mutual Information",

            "Logistic Regression coefficient magnitude",

            "Consensus ranking",

            "Spearman redundancy analysis",

            "Original training-data missingness",

            "Training-only 5-fold cross-validation"
        ],

        "final_feature_count": 4
    },

    # ------------------------------------------------
    # PREPROCESSING
    # ------------------------------------------------

    "preprocessing": {

        "scaler":
            "MinMaxScaler",

        "fit_data":
            "Training data only",

        "feature_range":
            "[0, pi]",

        "clip":
            True,

        "missing_values_after_preprocessing":
            0
    },

    # ------------------------------------------------
    # PATIENT SAFETY
    # ------------------------------------------------

    "patient_safety": {

        "train_test_patient_overlap":
            0,

        "one_selected_hour_per_patient":
            True
    },

    # ------------------------------------------------
    # NEXT PHASE
    # ------------------------------------------------

    "next_phase": (
        "Phase 6 — Quantum Encoding using the supplied "
        "4 model-ready scaled features."
    )
}

with open(
    QML_DATA_DIR / "qml_final_manifest.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        qml_final_manifest,
        f,
        indent=4
    )


# ===================================================
# 8. COPY PHASE 3 / PHASE 4 SUPPORTING FILES
# ===================================================

files_to_copy = [

    # Phase 3
    QML_DATA_DIR /
    "qml_phase3_reduced_data.joblib",

    QML_DATA_DIR /
    "qml_phase3_manifest.json",

    # Phase 4
    QML_DATA_DIR /
    "qml_phase4_selected_features.json"
]

print("\n=== CHECKING SUPPORTING QML FILES ===")

for source_file in files_to_copy:

    if source_file.exists():

        print(
            f"✓ Already present: "
            f"{source_file.name}"
        )

    else:

        print(
            f"⚠ Missing: "
            f"{source_file.name}"
        )


# ===================================================
# 9. NOTEBOOK INFORMATION
# ===================================================

PHASE1_5_NOTEBOOK = (
    PROJECT_DIR
    / "notebooks"
    / "06_QML_Phase1-5_DataPrep.ipynb"
)

PHASE6_10_NOTEBOOK = (
    PROJECT_DIR
    / "notebooks"
    / "07_QML_Phase6-10.ipynb"
)

print("\n=== NOTEBOOK CHECK ===")

if PHASE1_5_NOTEBOOK.exists():

    print(
        "✓ Phase 1–5 notebook:",
        PHASE1_5_NOTEBOOK
    )

else:

    print(
        "⚠ Phase 1–5 notebook not found:",
        PHASE1_5_NOTEBOOK
    )


if PHASE6_10_NOTEBOOK.exists():

    print(
        "✓ Phase 6–10 notebook:",
        PHASE6_10_NOTEBOOK
    )

else:

    print(
        "⚠ Phase 6–10 notebook not found:",
        PHASE6_10_NOTEBOOK
    )


# ===================================================
# 10. REQUIREMENTS FILE
# ===================================================

requirements_text = """qiskit==2.5.2
qiskit-machine-learning==0.9.1
numpy==2.2.6
pandas==2.3.3
scikit-learn==1.7.2
joblib==1.6.0
ipykernel==7.3.0
"""

with open(
    QML_DATA_DIR / "requirements_qml.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(requirements_text)


# ===================================================
# 11. README
# ===================================================

readme_text = """QML PHASE 1–5 HANDOFF

======================

Project:
Time-Series Analysis and Early Warning of Sepsis in ICU Patients
Using Smart Advanced Classifiers


PROJECT STRUCTURE
-----------------

QML data files are stored in:

data/processed/qml/

Phase 1–5 notebook:

notebooks/06_QML_Phase1-5_DataPrep.ipynb

Phase 6–10 notebook:

notebooks/07_QML_Phase6-10.ipynb


COMPLETED
---------

QML Phases 1–5 are complete.

Phase 1:
Qiskit/Qiskit Machine Learning environment and VQC stack selected.

Phase 2:
Review 1 patient-safe classical reference dataset validated.

Phase 3:
Patient-aware QML dataset reduction completed.

Phase 4:
Feature selection completed.

Phase 5:
Final quantum-compatible data preparation completed.


FINAL MODEL INPUT
-----------------

Training:

1000 rows
1000 unique patients
500 Class 0
500 Class 1

Testing:

5000 rows
5000 unique patients
4916 Class 0
84 Class 1

Positive prevalence:

1.68%

Train/test patient overlap:

0


SELECTED FEATURES
-----------------

1. ICULOS
2. Resp
3. HR
4. MAP


MISSING VALUES
--------------

0


SCALING
-------

MinMaxScaler fitted ONLY on training data.

Final feature range = [0, pi].

Test values outside training ranges were clipped
to the training-derived scaling boundaries.


MAIN DATA PACKAGE
-----------------

qml_final_data.joblib

Main objects:

X_train_scaled
X_test_scaled
y_train
y_test
selected_features
train_metadata
test_metadata


CSV ALTERNATIVES
----------------

qml_train_scaled.csv
qml_test_scaled.csv

qml_train_raw_selected.csv
qml_test_raw_selected.csv

qml_train_metadata.csv
qml_test_metadata.csv


IMPORTANT — DO NOT
------------------

Do NOT:

- create a new train/test split
- rebalance the supplied data again
- use SMOTE
- duplicate minority-class observations
- impute the data again
- perform feature selection again
- refit the supplied quantum scaler
- mix train and test patients


REMAINING QML WORK
------------------

Phase 6 — Quantum Encoding

- Use 4 input features
- Begin with a 4-qubit representation
- Select and implement an appropriate angle/rotation feature map
- Visualize and verify the encoding circuit

Phase 7 — Variational Quantum Classifier

- Configure feature map
- Configure ansatz
- Select entanglement structure
- Select optimizer
- Select circuit repetitions/depth
- Configure simulator
- Preserve reproducibility

Phase 8 — QML Training and Evaluation

- Train on supplied 1000 x 4 training data
- Evaluate on supplied 5000 x 4 test data
- Report Accuracy
- Precision
- Recall / Sensitivity
- Specificity
- F1
- Confusion matrix
- ROC-AUC / AUPRC where suitable
- Record training time, iterations, circuit details and limitations

Phase 9 — Fair Classical Benchmark

- Train a classical classifier using the EXACT same rows
  and EXACT same four features.

Phase 10 — Classical vs Quantum Comparison

- Compare predictive performance
- Training time
- Computational requirements
- Scalability
- Complexity
- Interpretability
- Qubit/circuit requirements
- Limitations


RANDOM SEED
-----------

42
"""

with open(
    QML_DATA_DIR / "README.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(readme_text)


# ===================================================
# 12. FINAL VALIDATION
# ===================================================

print("\n========================================")
print("      QML HANDOFF EXPORT COMPLETE")
print("========================================")

print("\nQML directory:")
print(QML_DATA_DIR)

print("\nTraining labels:")
print(
    y_qml_train_final
    .value_counts()
    .sort_index()
)

print("\nTesting labels:")
print(
    y_qml_test_final
    .value_counts()
    .sort_index()
)

print("\nSelected features:")
print(FINAL_QML_FEATURES)

print("\nFiles currently in data/processed/qml/:")

for file in sorted(QML_DATA_DIR.iterdir()):

    print(
        f"- {file.name}"
    )

print("\n✓ Phase 1–5 QML handoff is ready.")

QML data directory:
d:\SepsiChronos HDA\data\processed\qml

=== CHECKING SUPPORTING QML FILES ===
✓ Already present: qml_phase3_reduced_data.joblib
✓ Already present: qml_phase3_manifest.json
✓ Already present: qml_phase4_selected_features.json

=== NOTEBOOK CHECK ===
✓ Phase 1–5 notebook: d:\SepsiChronos HDA\notebooks\06_QML_Phase1-5_DataPrep.ipynb
✓ Phase 6–10 notebook: d:\SepsiChronos HDA\notebooks\07_QML_Phase6-10.ipynb

      QML HANDOFF EXPORT COMPLETE

QML directory:
d:\SepsiChronos HDA\data\processed\qml

Training labels:
SepsisLabel
0    500
1    500
Name: count, dtype: int64

Testing labels:
SepsisLabel
0    4916
1      84
Name: count, dtype: int64

Selected features:
['ICULOS', 'Resp', 'HR', 'MAP']

Files currently in data/processed/qml/:
- qml_final_data.joblib
- qml_final_manifest.json
- qml_phase3_manifest.json
- qml_phase3_reduced_data.joblib
- qml_phase4_selected_features.json
- qml_quantum_scaler.joblib
- qml_test_metadata.csv
- qml_test_raw_selected.csv
- qml_test_sca

In [41]:
# QML HANDOFF — COMPLETE INTEGRITY CHECK

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd


# ===================================================
# 1. PROJECT PATHS
# ===================================================

print("=" * 60)
print("QML PROJECT INTEGRITY CHECK")
print("=" * 60)

print("\nProject root:")
print(PROJECT_DIR)

QML_DIR = PROJECT_DIR / "data" / "processed" / "qml"


# ===================================================
# 2. REQUIRED PROJECT FILES
# ===================================================

required_files = [
    PROJECT_DIR / "notebooks" / "06_QML_Phase1-5_DataPrep.ipynb",
    PROJECT_DIR / "notebooks" / "07_QML_Phase6-10.ipynb",

    QML_DIR / "qml_final_data.joblib",
    QML_DIR / "qml_final_manifest.json",
    QML_DIR / "qml_phase3_reduced_data.joblib",
    QML_DIR / "qml_phase3_manifest.json",
    QML_DIR / "qml_phase4_selected_features.json",
    QML_DIR / "qml_quantum_scaler.joblib",

    QML_DIR / "qml_train_scaled.csv",
    QML_DIR / "qml_test_scaled.csv",
    QML_DIR / "qml_train_raw_selected.csv",
    QML_DIR / "qml_test_raw_selected.csv",

    QML_DIR / "qml_train_metadata.csv",
    QML_DIR / "qml_test_metadata.csv",

    QML_DIR / "README.txt",
    QML_DIR / "requirements_qml.txt"
]

print("\n=== REQUIRED FILE CHECK ===")

missing_files = []

for file_path in required_files:

    if file_path.exists():
        print(f"✓ {file_path.relative_to(PROJECT_DIR)}")
    else:
        print(f"✗ MISSING: {file_path.relative_to(PROJECT_DIR)}")
        missing_files.append(file_path)


# ===================================================
# 3. LOAD FINAL DATA PACKAGE
# ===================================================

print("\n=== LOADING FINAL DATA PACKAGE ===")

final_data = joblib.load(
    QML_DIR / "qml_final_data.joblib"
)

print("✓ qml_final_data.joblib loaded")


# ===================================================
# 4. CHECK REQUIRED KEYS
# ===================================================

expected_keys = [
    "X_train_scaled",
    "X_test_scaled",
    "X_train_raw_selected",
    "X_test_raw_selected",
    "y_train",
    "y_test",
    "selected_features",
    "train_metadata",
    "test_metadata",
    "random_seed",
    "quantum_feature_range"
]

print("\n=== FINAL DATA KEYS ===")

missing_keys = []

for key in expected_keys:

    if key in final_data:
        print(f"✓ {key}")
    else:
        print(f"✗ MISSING: {key}")
        missing_keys.append(key)


# ===================================================
# 5. SHAPES
# ===================================================

X_train = final_data["X_train_scaled"]
X_test = final_data["X_test_scaled"]

X_train_raw = final_data["X_train_raw_selected"]
X_test_raw = final_data["X_test_raw_selected"]

y_train = pd.Series(final_data["y_train"])
y_test = pd.Series(final_data["y_test"])

train_metadata = final_data["train_metadata"]
test_metadata = final_data["test_metadata"]

selected_features = list(
    final_data["selected_features"]
)

print("\n=== DATA SHAPES ===")

print("X_train_scaled :", X_train.shape)
print("X_test_scaled  :", X_test.shape)

print("X_train_raw    :", X_train_raw.shape)
print("X_test_raw     :", X_test_raw.shape)

print("y_train        :", y_train.shape)
print("y_test         :", y_test.shape)

print("Train metadata :", train_metadata.shape)
print("Test metadata  :", test_metadata.shape)


# ===================================================
# 6. EXPECTED DIMENSIONS
# ===================================================

print("\n=== DIMENSION CHECK ===")

assert X_train.shape == (1000, 4)
assert X_test.shape == (5000, 4)

assert X_train_raw.shape == (1000, 4)
assert X_test_raw.shape == (5000, 4)

assert len(y_train) == 1000
assert len(y_test) == 5000

assert len(train_metadata) == 1000
assert len(test_metadata) == 5000

print("✓ All dimensions are correct")


# ===================================================
# 7. CLASS DISTRIBUTION
# ===================================================

print("\n=== CLASS DISTRIBUTION ===")

print("\nTraining:")
print(y_train.value_counts().sort_index())

print("\nTesting:")
print(y_test.value_counts().sort_index())

assert y_train.value_counts().get(0, 0) == 500
assert y_train.value_counts().get(1, 0) == 500

assert y_test.value_counts().get(0, 0) == 4916
assert y_test.value_counts().get(1, 0) == 84

print("\n✓ Class distributions are correct")


# ===================================================
# 8. SELECTED FEATURES
# ===================================================

print("\n=== SELECTED FEATURES ===")

print(selected_features)

assert len(selected_features) == 4

print("✓ Exactly 4 features selected")


# ===================================================
# 9. MISSING VALUES
# ===================================================

print("\n=== MISSING VALUE CHECK ===")

print(
    "Train scaled missing:",
    int(X_train.isna().sum().sum())
)

print(
    "Test scaled missing :",
    int(X_test.isna().sum().sum())
)

print(
    "Train raw missing   :",
    int(X_train_raw.isna().sum().sum())
)

print(
    "Test raw missing    :",
    int(X_test_raw.isna().sum().sum())
)

assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

print("✓ Final scaled datasets contain no missing values")


# ===================================================
# 10. QUANTUM RANGE CHECK
# ===================================================

print("\n=== QUANTUM FEATURE RANGE CHECK ===")

train_min = X_train.min().min()
train_max = X_train.max().max()

test_min = X_test.min().min()
test_max = X_test.max().max()

print("Train minimum:", train_min)
print("Train maximum:", train_max)

print("Test minimum :", test_min)
print("Test maximum :", test_max)

print("Expected range: [0, pi]")
print("pi =", np.pi)

assert train_min >= 0
assert train_max <= np.pi + 1e-10

assert test_min >= 0
assert test_max <= np.pi + 1e-10

print("✓ All quantum-ready values are inside [0, pi]")


# ===================================================
# 11. TRAIN / TEST PATIENT OVERLAP
# ===================================================

print("\n=== PATIENT OVERLAP CHECK ===")

if (
    "Patient_Key" in train_metadata.columns
    and
    "Patient_Key" in test_metadata.columns
):

    train_patients = set(
        train_metadata["Patient_Key"]
    )

    test_patients = set(
        test_metadata["Patient_Key"]
    )

    overlap = train_patients.intersection(
        test_patients
    )

    print("Training patients:", len(train_patients))
    print("Testing patients :", len(test_patients))
    print("Overlap          :", len(overlap))

    assert len(overlap) == 0

    print("✓ Zero train/test patient overlap")

else:

    print(
        "⚠ Patient_Key column not available "
        "in final metadata."
    )


# ===================================================
# 12. ONE ROW PER PATIENT
# ===================================================

print("\n=== ONE-ROW-PER-PATIENT CHECK ===")

if "Patient_Key" in train_metadata.columns:

    train_counts = (
        train_metadata["Patient_Key"]
        .value_counts()
    )

    test_counts = (
        test_metadata["Patient_Key"]
        .value_counts()
    )

    print(
        "Maximum train observations per patient:",
        train_counts.max()
    )

    print(
        "Maximum test observations per patient:",
        test_counts.max()
    )

    assert train_counts.max() == 1
    assert test_counts.max() == 1

    print("✓ Exactly one selected observation per patient")


# ===================================================
# 13. LABEL / METADATA CONSISTENCY
# ===================================================

print("\n=== LABEL / METADATA CONSISTENCY ===")

if "SepsisLabel" in train_metadata.columns:

    metadata_train_labels = (
        train_metadata["SepsisLabel"]
        .astype(int)
        .reset_index(drop=True)
    )

    metadata_test_labels = (
        test_metadata["SepsisLabel"]
        .astype(int)
        .reset_index(drop=True)
    )

    assert np.array_equal(
        y_train.to_numpy(),
        metadata_train_labels.to_numpy()
    )

    assert np.array_equal(
        y_test.to_numpy(),
        metadata_test_labels.to_numpy()
    )

    print("✓ Training labels match metadata")
    print("✓ Testing labels match metadata")

else:

    print(
        "⚠ SepsisLabel not found in metadata."
    )


# ===================================================
# 14. RANDOM SEED
# ===================================================

print("\n=== REPRODUCIBILITY ===")

print(
    "Stored random seed:",
    final_data["random_seed"]
)

assert final_data["random_seed"] == 42

print("✓ Random seed = 42")


# ===================================================
# 15. MANIFEST CHECK
# ===================================================

print("\n=== MANIFEST CHECK ===")

with open(
    QML_DIR / "qml_final_manifest.json",
    "r",
    encoding="utf-8"
) as f:

    manifest = json.load(f)

print(
    "Manifest training:",
    manifest["training"]
)

print(
    "Manifest testing:",
    manifest["testing"]
)

assert manifest["training"]["rows"] == 1000
assert manifest["training"]["class_0"] == 500
assert manifest["training"]["class_1"] == 500

assert manifest["testing"]["rows"] == 5000
assert manifest["testing"]["class_0"] == 4916
assert manifest["testing"]["class_1"] == 84

print("✓ Manifest matches actual dataset")


# ===================================================
# 16. QUANTUM SCALER CHECK
# ===================================================

print("\n=== QUANTUM SCALER CHECK ===")

quantum_scaler = joblib.load(
    QML_DIR / "qml_quantum_scaler.joblib"
)

print(
    "Scaler type:",
    type(quantum_scaler).__name__
)

if hasattr(quantum_scaler, "feature_range"):
    print(
        "Feature range:",
        quantum_scaler.feature_range
    )

print("✓ Quantum scaler loaded successfully")


# ===================================================
# 17. PHASE 3 PACKAGE CHECK
# ===================================================

print("\n=== PHASE 3 PACKAGE CHECK ===")

phase3_data = joblib.load(
    QML_DIR / "qml_phase3_reduced_data.joblib"
)

phase3_y_train = pd.Series(
    phase3_data["y_train"]
)

phase3_y_test = pd.Series(
    phase3_data["y_test"]
)

print("Phase 3 train shape:",
      phase3_data["X_train_40"].shape)

print("Phase 3 test shape :",
      phase3_data["X_test_40"].shape)

print("\nPhase 3 train labels:")
print(
    phase3_y_train
    .value_counts()
    .sort_index()
)

print("\nPhase 3 test labels:")
print(
    phase3_y_test
    .value_counts()
    .sort_index()
)

assert phase3_y_train.value_counts().get(0, 0) == 500
assert phase3_y_train.value_counts().get(1, 0) == 500

assert phase3_y_test.value_counts().get(0, 0) == 4916
assert phase3_y_test.value_counts().get(1, 0) == 84

print("\n✓ Phase 3 labels are also correct")


# ===================================================
# 18. FINAL SUMMARY
# ===================================================

print("\n")
print("=" * 60)
print("FINAL QML INTEGRITY SUMMARY")
print("=" * 60)

checks = {
    "Required files": len(missing_files) == 0,
    "Required data keys": len(missing_keys) == 0,
    "Dataset dimensions": True,
    "Training classes": True,
    "Testing classes": True,
    "4 selected features": True,
    "No final missing values": True,
    "Quantum range [0, pi]": True,
    "Zero patient overlap": True,
    "One row per patient": True,
    "Label/metadata consistency": True,
    "Random seed = 42": True,
    "Manifest consistency": True,
    "Quantum scaler": True,
    "Phase 3 package": True
}

for check_name, status in checks.items():

    print(
        f"{'✓' if status else '✗'} {check_name}"
    )

print("\n" + "=" * 60)

if (
    len(missing_files) == 0
    and
    len(missing_keys) == 0
):

    print("✓ QML PIPELINE IS CONSISTENT AND READY FOR PHASE 6–10")

else:

    print("⚠ QML PIPELINE HAS ITEMS THAT NEED ATTENTION")

print("=" * 60)

QML PROJECT INTEGRITY CHECK

Project root:
d:\SepsiChronos HDA

=== REQUIRED FILE CHECK ===
✓ notebooks\06_QML_Phase1-5_DataPrep.ipynb
✓ notebooks\07_QML_Phase6-10.ipynb
✓ data\processed\qml\qml_final_data.joblib
✓ data\processed\qml\qml_final_manifest.json
✓ data\processed\qml\qml_phase3_reduced_data.joblib
✓ data\processed\qml\qml_phase3_manifest.json
✓ data\processed\qml\qml_phase4_selected_features.json
✓ data\processed\qml\qml_quantum_scaler.joblib
✓ data\processed\qml\qml_train_scaled.csv
✓ data\processed\qml\qml_test_scaled.csv
✓ data\processed\qml\qml_train_raw_selected.csv
✓ data\processed\qml\qml_test_raw_selected.csv
✓ data\processed\qml\qml_train_metadata.csv
✓ data\processed\qml\qml_test_metadata.csv
✓ data\processed\qml\README.txt
✓ data\processed\qml\requirements_qml.txt

=== LOADING FINAL DATA PACKAGE ===
✓ qml_final_data.joblib loaded

=== FINAL DATA KEYS ===
✓ X_train_scaled
✓ X_test_scaled
✓ X_train_raw_selected
✓ X_test_raw_selected
✓ y_train
✓ y_test
✓ selected_fea